In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
Firms = "/content/drive/MyDrive/India_Firms"
import pandas as pd

In [3]:
df = pd.read_parquet(f"{Firms}/India_Firms.parquet")

In [4]:
import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from pathlib import Path
import gc
import pickle



In [5]:

SPATIAL_THRESHOLD_M = 1000       # 1 km
TEMPORAL_THRESHOLD_H = 12        # 12 hours
TEMPORAL_THRESHOLD_S = 12 * 3600

# Output directory
OUTPUT_DIR = Path("fire_clustering_output")
OUTPUT_DIR.mkdir(exist_ok=True)

LOCAL_CLUSTER_DIR = OUTPUT_DIR / "local_clusters"
LOCAL_CLUSTER_DIR.mkdir(exist_ok=True)

EDGE_DIR = OUTPUT_DIR / "cross_day_edges"
EDGE_DIR.mkdir(exist_ok=True)

In [7]:
required_columns = [
    "latitude",
    "longitude",
    "datetime",
    "date",
    "x",
    "y",
    "satellite",
    "frp"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required columns are present.")


All required columns are present.


In [8]:
def cluster_one_day(day_df, date_value):
    """
    Cluster detections within one calendar day.

    Direct connection:
        distance <= 1000 m
        AND
        time difference <= 12 h

    Returns
    -------
    day_df : detection-level dataframe
    cluster_summary : cluster-level dataframe
    """

    day_df = day_df.copy()

    if len(day_df) == 0:
        return day_df, pd.DataFrame()

    # --------------------------------------------------------
    # Ensure clean ordering
    # --------------------------------------------------------

    day_df = day_df.reset_index(drop=True)

    # --------------------------------------------------------
    # Coordinates
    # --------------------------------------------------------

    coords = day_df[["x", "y"]].to_numpy(dtype=np.float64)

    # --------------------------------------------------------
    # Time in seconds
    # --------------------------------------------------------

    times = (
        day_df["datetime"]
        .astype("int64")
        .to_numpy()
        / 1e9
    )

    # --------------------------------------------------------
    # KD-tree spatial search
    # --------------------------------------------------------

    tree = cKDTree(coords)

    pairs = tree.query_pairs(
        r=SPATIAL_THRESHOLD_M,
        output_type="ndarray"
    )

    print(
        f"{date_value}: "
        f"{len(day_df):,} detections, "
        f"{len(pairs):,} spatial pairs"
    )

    # --------------------------------------------------------
    # Temporal filtering
    # --------------------------------------------------------

    if len(pairs) > 0:

        dt = np.abs(
            times[pairs[:, 0]]
            -
            times[pairs[:, 1]]
        )

        valid = dt <= TEMPORAL_THRESHOLD_S

        valid_pairs = pairs[valid]

    else:

        valid_pairs = np.empty(
            (0, 2),
            dtype=np.int64
        )

    print(
        f"{date_value}: "
        f"{len(valid_pairs):,} valid pairs"
    )

    # --------------------------------------------------------
    # Connected components
    # --------------------------------------------------------

    n = len(day_df)

    if len(valid_pairs) > 0:

        rows = np.concatenate([
            valid_pairs[:, 0],
            valid_pairs[:, 1]
        ])

        cols = np.concatenate([
            valid_pairs[:, 1],
            valid_pairs[:, 0]
        ])

        data = np.ones(
            len(rows),
            dtype=np.uint8
        )

        graph = coo_matrix(
            (data, (rows, cols)),
            shape=(n, n)
        ).tocsr()

        n_clusters, labels = connected_components(
            graph,
            directed=False
        )

    else:

        # Every detection becomes its own cluster
        n_clusters = n

        labels = np.arange(
            n,
            dtype=np.int64
        )

    print(
        f"{date_value}: "
        f"{n_clusters:,} local clusters"
    )

    # --------------------------------------------------------
    # Globally unique local cluster IDs
    # --------------------------------------------------------

    date_string = pd.Timestamp(date_value).strftime("%Y%m%d")

    local_cluster_ids = np.array(
        [
            f"{date_string}_{label}"
            for label in labels
        ],
        dtype=object
    )

    day_df["local_cluster_id"] = local_cluster_ids

    # --------------------------------------------------------
    # Cluster summary
    # --------------------------------------------------------

    grouped = day_df.groupby(
        "local_cluster_id",
        sort=False
    )

    cluster_summary = grouped.agg(
        start_time=("datetime", "min"),
        end_time=("datetime", "max"),
        n_detections=("detection_id", "size"),
        centroid_x=("x", "mean"),
        centroid_y=("y", "mean"),
        total_frp=("frp", "sum"),
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        n_observation_times=("datetime", "nunique")
    ).reset_index()

    cluster_summary["duration_hours"] = (
        (
            cluster_summary["end_time"]
            -
            cluster_summary["start_time"]
        )
        .dt.total_seconds()
        / 3600
    )

    # --------------------------------------------------------
    # Satellite information
    # --------------------------------------------------------

    satellite_summary = (
        grouped["satellite"]
        .agg(
            lambda x: ",".join(
                sorted(
                    x.dropna()
                    .astype(str)
                    .unique()
                )
            )
        )
        .rename("satellites")
        .reset_index()
    )

    cluster_summary = cluster_summary.merge(
        satellite_summary,
        on="local_cluster_id",
        how="left"
    )

    cluster_summary["n_satellites"] = (
        cluster_summary["satellites"]
        .str.split(",")
        .str.len()
    )

    # --------------------------------------------------------
    # Maximum radius from centroid
    # --------------------------------------------------------

    centroid_lookup = cluster_summary[
        [
            "local_cluster_id",
            "centroid_x",
            "centroid_y"
        ]
    ].set_index("local_cluster_id")

    cx = (
        day_df["local_cluster_id"]
        .map(centroid_lookup["centroid_x"])
        .to_numpy()
    )

    cy = (
        day_df["local_cluster_id"]
        .map(centroid_lookup["centroid_y"])
        .to_numpy()
    )

    distances = np.sqrt(
        (day_df["x"].to_numpy() - cx) ** 2
        +
        (day_df["y"].to_numpy() - cy) ** 2
    )

    day_df["_distance_from_centroid"] = distances

    max_radius = (
        day_df
        .groupby("local_cluster_id")["_distance_from_centroid"]
        .max()
        .rename("max_radius_m")
        .reset_index()
    )

    cluster_summary = cluster_summary.merge(
        max_radius,
        on="local_cluster_id",
        how="left"
    )

    cluster_summary["max_radius_km"] = (
        cluster_summary["max_radius_m"] / 1000
    )

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return day_df, cluster_summary


In [12]:
df = df.reset_index(drop=True)

df["detection_id"] = np.arange(
    len(df),
    dtype=np.int64
)


In [13]:
test_date = pd.Timestamp("2021-03-31").date()

test_day = df[
    df["date"] == test_date
].copy()

test_result, test_clusters = cluster_one_day(
    test_day,
    test_date
)


2021-03-31: 37,801 detections, 89,322 spatial pairs
2021-03-31: 75,626 valid pairs
2021-03-31: 12,641 local clusters


In [14]:
print(test_result.shape)
print(test_clusters.shape)

print(
    test_clusters.head()
)


(37801, 25)
(12641, 15)
  local_cluster_id          start_time            end_time  n_detections  \
0       20210331_0 2021-03-31 07:12:00 2021-03-31 08:02:00             3   
1       20210331_1 2021-03-31 07:12:00 2021-03-31 20:37:00            23   
2       20210331_2 2021-03-31 07:12:00 2021-03-31 08:02:00             4   
3       20210331_3 2021-03-31 07:12:00 2021-03-31 08:53:00             4   
4       20210331_4 2021-03-31 07:12:00 2021-03-31 07:12:00             1   

     centroid_x    centroid_y  total_frp   mean_frp  max_frp  \
0  7.487883e+06  1.311624e+06      54.28  18.093333    24.41   
1  7.486238e+06  1.306968e+06     225.58   9.807826    23.55   
2  7.523222e+06  1.236856e+06      28.67   7.167500    10.26   
3  7.621429e+06  1.428727e+06      17.72   4.430000     8.70   
4  7.572603e+06  1.275997e+06       5.83   5.830000     5.83   

   n_observation_times  duration_hours satellites  n_satellites  max_radius_m  \
0                    2        0.833333   N20,SNPP    

In [16]:
# ============================================================
# STEP 7 — Convert cluster centroids to latitude/longitude
# ============================================================

centroid_gdf = gpd.GeoDataFrame(
    test_clusters.copy(),
    geometry=gpd.points_from_xy(
        test_clusters["centroid_x"],
        test_clusters["centroid_y"]
    ),
    crs="EPSG:6933"
)

# Convert projected coordinates → WGS84
centroid_gdf = centroid_gdf.to_crs("EPSG:4326")

# Extract longitude and latitude
test_clusters["centroid_lon"] = centroid_gdf.geometry.x
test_clusters["centroid_lat"] = centroid_gdf.geometry.y

print(test_clusters[
    [
        "local_cluster_id",
        "centroid_x",
        "centroid_y",
        "centroid_lon",
        "centroid_lat"
    ]
].head())


  local_cluster_id    centroid_x    centroid_y  centroid_lon  centroid_lat
0       20210331_0  7.487883e+06  1.311624e+06     77.605677     10.335827
1       20210331_1  7.486238e+06  1.306968e+06     77.588626     10.298747
2       20210331_2  7.523222e+06  1.236856e+06     77.971940      9.740880
3       20210331_3  7.621429e+06  1.428727e+06     78.989767     11.269882
4       20210331_4  7.572603e+06  1.275997e+06     78.483730     10.052200


In [17]:
print(
    "Latitude range:",
    test_clusters["centroid_lat"].min(),
    test_clusters["centroid_lat"].max()
)

print(
    "Longitude range:",
    test_clusters["centroid_lon"].min(),
    test_clusters["centroid_lon"].max()
)

Latitude range: 9.385064999998278 33.84174
Longitude range: 68.71513999999999 93.62148499999999


In [19]:
date_a = pd.Timestamp("2021-03-31").date()
date_b = pd.Timestamp("2021-04-01").date()

day_a = df[
    df["date"] == date_a
].copy()

day_b = df[
    df["date"] == date_b
].copy()


In [20]:
clustered_b, summary_b = cluster_one_day(
    day_b,
    date_b
)


2021-04-01: 36,359 detections, 90,202 spatial pairs
2021-04-01: 74,095 valid pairs
2021-04-01: 12,901 local clusters


In [24]:
def cross_day_cluster_edges(
    day_a,
    day_b,
    date_a,
    date_b
):
    """
    Find unique local-cluster connections between two dates.

    A cross-day connection requires:
        distance <= 1000 m
        AND
        time difference <= 12 hours

    Only the relevant temporal boundary portions are searched:
        previous day: last 12 hours
        next day:     first 12 hours

    Returns
    -------
    pandas.DataFrame
        Unique local-cluster-to-local-cluster edges.
    """

    # ========================================================
    # 1. Define the temporal boundary
    # ========================================================

    midnight_b = pd.Timestamp(date_b)

    boundary_a_start = (
        midnight_b - pd.Timedelta(hours=12)
    )

    boundary_b_end = (
        midnight_b + pd.Timedelta(hours=12)
    )

    # ========================================================
    # 2. Select only detections that could possibly connect
    # ========================================================

    a_boundary = day_a[
        day_a["datetime"] >= boundary_a_start
    ].copy()

    b_boundary = day_b[
        day_b["datetime"] < boundary_b_end
    ].copy()

    print(
        f"{date_a}: "
        f"{len(a_boundary):,} boundary detections"
    )

    print(
        f"{date_b}: "
        f"{len(b_boundary):,} boundary detections"
    )

    # ========================================================
    # 3. If either side is empty
    # ========================================================

    if (
        len(a_boundary) == 0
        or
        len(b_boundary) == 0
    ):

        return pd.DataFrame(
            columns=[
                "local_cluster_id_a",
                "local_cluster_id_b"
            ]
        )

    # ========================================================
    # 4. Build KD-tree for Day B
    # ========================================================

    coords_b = b_boundary[
        ["x", "y"]
    ].to_numpy(
        dtype=np.float64
    )

    tree_b = cKDTree(coords_b)

    # ========================================================
    # 5. Coordinates for Day A
    # ========================================================

    coords_a = a_boundary[
        ["x", "y"]
    ].to_numpy(
        dtype=np.float64
    )

    # ========================================================
    # 6. Spatial candidate search
    # ========================================================

    spatial_neighbors = tree_b.query_ball_point(
        coords_a,
        r=SPATIAL_THRESHOLD_M
    )

    # ========================================================
    # 7. Convert timestamps to seconds
    # ========================================================

    time_a = (
        a_boundary["datetime"]
        .astype("int64")
        .to_numpy()
        / 1e9
    )

    time_b = (
        b_boundary["datetime"]
        .astype("int64")
        .to_numpy()
        / 1e9
    )

    # ========================================================
    # 8. Generate cluster-level edges
    # ========================================================

    edge_a = []
    edge_b = []

    for i, neighbors in enumerate(
        spatial_neighbors
    ):

        if len(neighbors) == 0:
            continue

        neighbors = np.asarray(
            neighbors,
            dtype=np.int64
        )

        # Temporal difference
        dt = np.abs(
            time_a[i]
            -
            time_b[neighbors]
        )

        # Keep <= 12 hour connections
        valid_neighbors = neighbors[
            dt <= TEMPORAL_THRESHOLD_S
        ]

        if len(valid_neighbors) == 0:
            continue

        cluster_a = (
            a_boundary.iloc[i]
            ["local_cluster_id"]
        )

        cluster_b = (
            b_boundary.iloc[
                valid_neighbors
            ]["local_cluster_id"]
            .to_numpy()
        )

        edge_a.extend(
            [cluster_a] * len(cluster_b)
        )

        edge_b.extend(
            cluster_b.tolist()
        )

    # ========================================================
    # 9. No edges
    # ========================================================

    if len(edge_a) == 0:

        return pd.DataFrame(
            columns=[
                "local_cluster_id_a",
                "local_cluster_id_b"
            ]
        )

    # ========================================================
    # 10. Create edge table
    # ========================================================

    edges = pd.DataFrame({
        "local_cluster_id_a": edge_a,
        "local_cluster_id_b": edge_b
    })

    # ========================================================
    # 11. Remove duplicate cluster-level edges
    # ========================================================

    edges = edges.drop_duplicates(
        ignore_index=True
    )

    return edges



In [25]:
print(cross_day_cluster_edges)


<function cross_day_cluster_edges at 0x78b8935e5940>


In [26]:
date_a = pd.Timestamp("2021-03-31").date()
date_b = pd.Timestamp("2021-04-01").date()


In [29]:
# ============================================================
# MARCH 31 → APRIL 1 CROSS-DAY TEST
# ============================================================

date_a = pd.Timestamp("2021-03-31").date()
date_b = pd.Timestamp("2021-04-01").date()

# ------------------------------------------------------------
# Get March 31
# ------------------------------------------------------------

day_a = df[
    df["date"] == date_a
].copy()

print(
    f"{date_a}: {len(day_a):,} detections"
)

# ------------------------------------------------------------
# Get April 1
# ------------------------------------------------------------

day_b = df[
    df["date"] == date_b
].copy()

print(
    f"{date_b}: {len(day_b):,} detections"
)

# ------------------------------------------------------------
# Cluster March 31
# ------------------------------------------------------------

clustered_a, summary_a = cluster_one_day(
    day_a,
    date_a
)

# ------------------------------------------------------------
# Cluster April 1
# ------------------------------------------------------------

clustered_b, summary_b = cluster_one_day(
    day_b,
    date_b
)

# ------------------------------------------------------------
# Cross-day cluster connections
# ------------------------------------------------------------

edges = cross_day_cluster_edges(
    clustered_a,
    clustered_b,
    date_a,
    date_b
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CROSS-DAY TEST RESULTS")
print("=" * 60)

print(
    f"March 31 detections: "
    f"{len(clustered_a):,}"
)

print(
    f"March 31 local clusters: "
    f"{summary_a['local_cluster_id'].nunique():,}"
)

print(
    f"April 1 detections: "
    f"{len(clustered_b):,}"
)

print(
    f"April 1 local clusters: "
    f"{summary_b['local_cluster_id'].nunique():,}"
)

print(
    f"Unique cross-day cluster edges: "
    f"{len(edges):,}"
)


2021-03-31: 37,801 detections
2021-04-01: 36,359 detections
2021-03-31: 37,801 detections, 89,322 spatial pairs
2021-03-31: 75,626 valid pairs
2021-03-31: 12,641 local clusters
2021-04-01: 36,359 detections, 90,202 spatial pairs
2021-04-01: 74,095 valid pairs
2021-04-01: 12,901 local clusters
2021-03-31: 14,628 boundary detections
2021-04-01: 21,957 boundary detections

CROSS-DAY TEST RESULTS
March 31 detections: 37,801
March 31 local clusters: 12,641
April 1 detections: 36,359
April 1 local clusters: 12,901
Unique cross-day cluster edges: 2,066


In [30]:
test_dates = pd.date_range(
    "2021-03-28",
    "2021-04-03",
    freq="D"
).date

print(test_dates)


[datetime.date(2021, 3, 28) datetime.date(2021, 3, 29)
 datetime.date(2021, 3, 30) datetime.date(2021, 3, 31)
 datetime.date(2021, 4, 1) datetime.date(2021, 4, 2)
 datetime.date(2021, 4, 3)]


In [32]:
# ============================================================
# STEP 9 — SEVEN-DAY LOCAL CLUSTERING
# ============================================================

daily_clustered = {}
daily_summaries = {}

for d in test_dates:

    day_df = df[
        df["date"] == d
    ].copy()

    if len(day_df) == 0:
        print(f"{d}: NO DATA")
        continue

    clustered_day, summary_day = cluster_one_day(
        day_df,
        d
    )

    daily_clustered[d] = clustered_day
    daily_summaries[d] = summary_day

    print(
        f"{d}: "
        f"{len(day_df):,} detections → "
        f"{len(summary_day):,} local clusters"
    )


2021-03-28: 24,105 detections, 43,813 spatial pairs
2021-03-28: 38,774 valid pairs
2021-03-28: 10,100 local clusters
2021-03-28: 24,105 detections → 10,100 local clusters
2021-03-29: 27,979 detections, 63,325 spatial pairs
2021-03-29: 54,783 valid pairs
2021-03-29: 10,803 local clusters
2021-03-29: 27,979 detections → 10,803 local clusters
2021-03-30: 33,403 detections, 78,414 spatial pairs
2021-03-30: 66,716 valid pairs
2021-03-30: 11,967 local clusters
2021-03-30: 33,403 detections → 11,967 local clusters
2021-03-31: 37,801 detections, 89,322 spatial pairs
2021-03-31: 75,626 valid pairs
2021-03-31: 12,641 local clusters
2021-03-31: 37,801 detections → 12,641 local clusters
2021-04-01: 36,359 detections, 90,202 spatial pairs
2021-04-01: 74,095 valid pairs
2021-04-01: 12,901 local clusters
2021-04-01: 36,359 detections → 12,901 local clusters
2021-04-02: 29,842 detections, 65,768 spatial pairs
2021-04-02: 54,690 valid pairs
2021-04-02: 10,922 local clusters
2021-04-02: 29,842 detection

In [33]:
# ============================================================
# STEP 10 — SEVEN-DAY CROSS-DAY CLUSTER EDGES
# ============================================================

all_cross_day_edges = []

for i in range(len(test_dates) - 1):

    date_a = test_dates[i]
    date_b = test_dates[i + 1]

    # Skip if one of the dates has no data
    if (
        date_a not in daily_clustered
        or
        date_b not in daily_clustered
    ):
        print(
            f"{date_a} → {date_b}: skipped"
        )
        continue

    print("\n" + "-" * 60)
    print(
        f"Processing: {date_a} → {date_b}"
    )
    print("-" * 60)

    edges_day = cross_day_cluster_edges(
        daily_clustered[date_a],
        daily_clustered[date_b],
        date_a,
        date_b
    )

    print(
        f"Cluster-level edges: "
        f"{len(edges_day):,}"
    )

    all_cross_day_edges.append(
        edges_day
    )



------------------------------------------------------------
Processing: 2021-03-28 → 2021-03-29
------------------------------------------------------------
2021-03-28: 7,883 boundary detections
2021-03-29: 17,859 boundary detections
Cluster-level edges: 845

------------------------------------------------------------
Processing: 2021-03-29 → 2021-03-30
------------------------------------------------------------
2021-03-29: 10,120 boundary detections
2021-03-30: 20,415 boundary detections
Cluster-level edges: 1,304

------------------------------------------------------------
Processing: 2021-03-30 → 2021-03-31
------------------------------------------------------------
2021-03-30: 12,988 boundary detections
2021-03-31: 23,173 boundary detections
Cluster-level edges: 1,744

------------------------------------------------------------
Processing: 2021-03-31 → 2021-04-01
------------------------------------------------------------
2021-03-31: 14,628 boundary detections
2021-04-01: 2

In [34]:
if len(all_cross_day_edges) > 0:

    all_cross_day_edges = pd.concat(
        all_cross_day_edges,
        ignore_index=True
    )

else:

    all_cross_day_edges = pd.DataFrame(
        columns=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )

print(
    "Total cross-day cluster edges:",
    len(all_cross_day_edges)
)


Total cross-day cluster edges: 8738


In [35]:
all_local_clusters = []

for d in test_dates:

    if d not in daily_summaries:
        continue

    ids = daily_summaries[d][
        "local_cluster_id"
    ].astype(str)

    all_local_clusters.extend(
        ids.tolist()
    )

all_local_clusters = pd.Index(
    all_local_clusters
).unique()

print(
    "Total local clusters:",
    len(all_local_clusters)
)


Total local clusters: 77244


In [36]:
cluster_to_node = pd.Series(
    np.arange(
        len(all_local_clusters),
        dtype=np.int64
    ),
    index=all_local_clusters
)

print(
    "Graph nodes:",
    len(cluster_to_node)
)


Graph nodes: 77244


In [37]:
edge_node_a = (
    all_cross_day_edges[
        "local_cluster_id_a"
    ]
    .map(cluster_to_node)
    .to_numpy(dtype=np.int64)
)

edge_node_b = (
    all_cross_day_edges[
        "local_cluster_id_b"
    ]
    .map(cluster_to_node)
    .to_numpy(dtype=np.int64)
)

print(
    "Graph edges:",
    len(edge_node_a)
)


Graph edges: 8738


In [38]:
print(
    "Missing A nodes:",
    np.sum(pd.isna(edge_node_a))
)

print(
    "Missing B nodes:",
    np.sum(pd.isna(edge_node_b))
)


Missing A nodes: 0
Missing B nodes: 0


In [39]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

n_nodes = len(all_local_clusters)

if len(edge_node_a) > 0:

    row = np.concatenate([
        edge_node_a,
        edge_node_b
    ])

    col = np.concatenate([
        edge_node_b,
        edge_node_a
    ])

    data = np.ones(
        len(row),
        dtype=np.uint8
    )

    graph = coo_matrix(
        (data, (row, col)),
        shape=(n_nodes, n_nodes)
    ).tocsr()

else:

    graph = coo_matrix(
        (n_nodes, n_nodes),
        dtype=np.uint8
    ).tocsr()

n_global_components, labels = (
    connected_components(
        graph,
        directed=False
    )
)

print(
    "Total local clusters:",
    n_nodes
)

print(
    "Total cross-day edges:",
    len(all_cross_day_edges)
)

print(
    "Global connected components:",
    n_global_components
)


Total local clusters: 77244
Total cross-day edges: 8738
Global connected components: 68514


In [44]:
global_cluster_map["date"] = (
    global_cluster_map[
        "local_cluster_id"
    ]
    .str[:8]
)

multi_day_events = (
    global_cluster_map
    .groupby("fire_event_id")["date"]
    .nunique()
)

print(
    "Global events:",
    len(multi_day_events)
)

print(
    "Events spanning >= 2 days:",
    (multi_day_events >= 2).sum()
)

print(
    "Events spanning >= 3 days:",
    (multi_day_events >= 3).sum()
)

print(
    "Maximum number of days spanned:",
    multi_day_events.max()
)


Global events: 68514
Events spanning >= 2 days: 5999
Events spanning >= 3 days: 785
Maximum number of days spanned: 7


In [42]:
# ============================================================
# STEP 18 — CREATE GLOBAL FIRE-EVENT IDs
# ============================================================

global_cluster_map = pd.DataFrame({
    "local_cluster_id": all_local_clusters,
    "global_component": labels
})

# Create readable global fire-event IDs
global_cluster_map["fire_event_id"] = (
    "FIRE_" +
    global_cluster_map["global_component"]
    .astype(str)
    .str.zfill(8)
)

print(global_cluster_map.head(20))

print(
    "\nNumber of global fire events:",
    global_cluster_map["fire_event_id"].nunique()
)


   local_cluster_id  global_component  fire_event_id
0        20210328_0                 0  FIRE_00000000
1        20210328_1                 1  FIRE_00000001
2        20210328_2                 2  FIRE_00000002
3        20210328_3                 3  FIRE_00000003
4        20210328_4                 4  FIRE_00000004
5        20210328_5                 5  FIRE_00000005
6        20210328_6                 6  FIRE_00000006
7        20210328_7                 7  FIRE_00000007
8        20210328_8                 8  FIRE_00000008
9        20210328_9                 9  FIRE_00000009
10      20210328_10                10  FIRE_00000010
11      20210328_11                11  FIRE_00000011
12      20210328_12                12  FIRE_00000012
13      20210328_13                13  FIRE_00000013
14      20210328_14                14  FIRE_00000014
15      20210328_15                15  FIRE_00000015
16      20210328_16                16  FIRE_00000016
17      20210328_17                17  FIRE_00

In [43]:
# ============================================================
# STEP 19 — EXTRACT LOCAL CLUSTER DATE
# ============================================================

global_cluster_map["date"] = pd.to_datetime(
    global_cluster_map["local_cluster_id"]
    .str[:8],
    format="%Y%m%d"
).dt.date

print(
    global_cluster_map.head()
)


  local_cluster_id  global_component  fire_event_id        date
0       20210328_0                 0  FIRE_00000000  2021-03-28
1       20210328_1                 1  FIRE_00000001  2021-03-28
2       20210328_2                 2  FIRE_00000002  2021-03-28
3       20210328_3                 3  FIRE_00000003  2021-03-28
4       20210328_4                 4  FIRE_00000004  2021-03-28


In [45]:
# ============================================================
# STEP 20 — MULTI-DAY EVENT VALIDATION
# ============================================================

multi_day_events = (
    global_cluster_map
    .groupby("fire_event_id")["date"]
    .nunique()
)

print(
    "Global fire events:",
    len(multi_day_events)
)

print(
    "Events spanning >= 2 days:",
    (multi_day_events >= 2).sum()
)

print(
    "Events spanning >= 3 days:",
    (multi_day_events >= 3).sum()
)

print(
    "Events spanning >= 4 days:",
    (multi_day_events >= 4).sum()
)

print(
    "Events spanning >= 5 days:",
    (multi_day_events >= 5).sum()
)

print(
    "Maximum number of days spanned:",
    multi_day_events.max()
)


Global fire events: 68514
Events spanning >= 2 days: 5999
Events spanning >= 3 days: 785
Events spanning >= 4 days: 199
Events spanning >= 5 days: 48
Maximum number of days spanned: 7


In [46]:
# ============================================================
# STEP 21 — INSPECT MULTI-DAY GLOBAL EVENTS
# ============================================================

multi_day_ids = multi_day_events[
    multi_day_events >= 2
].index

multi_day_example = (
    global_cluster_map[
        global_cluster_map["fire_event_id"]
        .isin(multi_day_ids)
    ]
    .sort_values(
        ["fire_event_id", "date"]
    )
)

print(
    multi_day_example.head(30)
)


      local_cluster_id  global_component  fire_event_id      date
52         20210328_52                52  FIRE_00000052  20210328
19486    20210329_9386                52  FIRE_00000052  20210329
57         20210328_57                57  FIRE_00000057  20210328
19762    20210329_9662                57  FIRE_00000057  20210329
103       20210328_103               103  FIRE_00000103  20210328
13303    20210329_3203               103  FIRE_00000103  20210329
24816    20210330_3913               103  FIRE_00000103  20210330
35906    20210331_3036               103  FIRE_00000103  20210331
117       20210328_117               117  FIRE_00000117  20210328
10375     20210329_275               117  FIRE_00000117  20210329
218       20210328_218               218  FIRE_00000218  20210328
10279     20210329_179               218  FIRE_00000218  20210329
19121    20210329_9021               218  FIRE_00000218  20210329
285       20210328_285               285  FIRE_00000285  20210328
14970    2

In [47]:
# ============================================================
# STEP 22 — GLOBAL EVENT DAY-SPAN DISTRIBUTION
# ============================================================

day_span_distribution = (
    multi_day_events
    .value_counts()
    .sort_index()
)

print(day_span_distribution)


date
1    62515
2     5214
3      586
4      151
5       32
6       11
7        5
Name: count, dtype: int64


In [49]:
# ============================================================
# STEP 23 — BUILD DATE INDEX
# ============================================================

date_index = {}

for d, group in df.groupby("date", sort=False):
    date_index[d] = group.index.to_numpy()

print(
    "Number of observed dates:",
    len(date_index)
)

print(
    "First date:",
    min(date_index.keys())
)

print(
    "Last date:",
    max(date_index.keys())
)


Number of observed dates: 5337
First date: 2012-01-20
Last date: 2026-08-31


In [50]:
# ============================================================
# STEP 24 — VERIFY MISSING CALENDAR DATES
# ============================================================

observed_dates = pd.DatetimeIndex(
    sorted(date_index.keys())
)

expected_dates = pd.date_range(
    observed_dates.min(),
    observed_dates.max(),
    freq="D"
)

missing_dates = expected_dates.difference(
    observed_dates
)

print(
    "Expected dates:",
    len(expected_dates)
)

print(
    "Observed dates:",
    len(observed_dates)
)

print(
    "Missing dates:",
    len(missing_dates)
)

print(
    missing_dates
)


Expected dates: 5338
Observed dates: 5337
Missing dates: 1
DatetimeIndex(['2012-03-25'], dtype='datetime64[ns]', freq='D')


In [51]:
# ============================================================
# STEP 25 — PRODUCTION DATE SEQUENCE
# ============================================================

production_dates = sorted(
    date_index.keys()
)

print(
    "Production dates:",
    len(production_dates)
)

print(
    "First 5:",
    production_dates[:5]
)

print(
    "Last 5:",
    production_dates[-5:]
)


Production dates: 5337
First 5: [datetime.date(2012, 1, 20), datetime.date(2012, 1, 21), datetime.date(2012, 1, 22), datetime.date(2012, 1, 23), datetime.date(2012, 1, 24)]
Last 5: [datetime.date(2026, 8, 27), datetime.date(2026, 8, 28), datetime.date(2026, 8, 29), datetime.date(2026, 8, 30), datetime.date(2026, 8, 31)]


In [52]:
# ============================================================
# STEP 26 — MEMORY-SAFE PRODUCTION TEMPORAL ENGINE
# ============================================================

import gc
import pandas as pd
import numpy as np


def process_temporal_range(
    combined_df,
    start_date,
    end_date,
    verbose=True
):
    """
    Memory-conscious temporal processing.

    For each observed date:
        1. Extract daily detections
        2. Create local clusters
        3. Store local cluster summary
        4. Retain only boundary detections
        5. Match with previous day's boundary
        6. Store cluster-level cross-day edges
        7. Release temporary objects

    Returns
    -------
    all_cluster_summaries : DataFrame
        One row per local cluster.

    all_cross_day_edges : DataFrame
        Unique cross-day local-cluster connections.
    """

    # --------------------------------------------------------
    # Select dates
    # --------------------------------------------------------

    mask = (
        (combined_df["date"] >= start_date)
        &
        (combined_df["date"] <= end_date)
    )

    selected_dates = sorted(
        combined_df.loc[mask, "date"].unique()
    )

    print("=" * 70)
    print("PRODUCTION TEMPORAL ENGINE")
    print("=" * 70)

    print(
        f"Date range: {start_date} → {end_date}"
    )

    print(
        f"Observed dates to process: "
        f"{len(selected_dates):,}"
    )

    # --------------------------------------------------------
    # Storage
    # --------------------------------------------------------

    cluster_summaries = []
    cross_day_edges = []

    # --------------------------------------------------------
    # Previous-day boundary information
    # --------------------------------------------------------

    previous_boundary = None
    previous_date = None

    # --------------------------------------------------------
    # Main temporal loop
    # --------------------------------------------------------

    for counter, current_date in enumerate(
        selected_dates,
        start=1
    ):

        if verbose:
            print(
                f"\n[{counter}/{len(selected_dates)}] "
                f"{current_date}"
            )

        # ====================================================
        # 1. Extract current day
        # ====================================================

        day_mask = (
            combined_df["date"] == current_date
        )

        day_df = combined_df.loc[
            day_mask
        ].copy()

        if len(day_df) == 0:

            print(
                f"{current_date}: no detections"
            )

            continue

        # ====================================================
        # 2. Local clustering
        # ====================================================

        clustered_day, summary_day = (
            cluster_one_day(
                day_df,
                current_date
            )
        )

        if verbose:
            print(
                f"{current_date}: "
                f"{len(day_df):,} detections → "
                f"{len(summary_day):,} local clusters"
            )

        # ====================================================
        # 3. Store local cluster summary
        # ====================================================

        cluster_summaries.append(
            summary_day.copy()
        )

        # ====================================================
        # 4. Prepare current-day boundary
        #
        # Only the FIRST 12 hours of the current date
        # can connect to the previous calendar date.
        # ====================================================

        current_midnight = pd.Timestamp(
            current_date
        )

        current_boundary_end = (
            current_midnight
            +
            pd.Timedelta(hours=12)
        )

        current_boundary = clustered_day[
            clustered_day["datetime"]
            <
            current_boundary_end
        ].copy()

        # ====================================================
        # 5. Cross-day matching
        # ====================================================

        if (
            previous_boundary is not None
            and
            previous_date is not None
        ):

            # -----------------------------------------------
            # IMPORTANT:
            # Only connect if these are actually consecutive
            # calendar dates.
            #
            # If a calendar date is missing, we do NOT try to
            # connect the two observed dates.
            # -----------------------------------------------

            expected_next_date = (
                previous_date
                +
                pd.Timedelta(days=1)
            )

            if current_date == expected_next_date:

                edges_day = cross_day_cluster_edges(
                    previous_boundary,
                    current_boundary,
                    previous_date,
                    current_date
                )

                if len(edges_day) > 0:

                    cross_day_edges.append(
                        edges_day
                    )

                if verbose:
                    print(
                        f"Cross-day cluster edges: "
                        f"{len(edges_day):,}"
                    )

            else:

                if verbose:
                    print(
                        f"No cross-day matching: "
                        f"{previous_date} → "
                        f"{current_date}"
                    )

        # ====================================================
        # 6. Current boundary becomes previous boundary
        # ====================================================

        previous_boundary = current_boundary
        previous_date = current_date

        # ====================================================
        # 7. Release temporary objects
        # ====================================================

        del day_df
        del clustered_day
        del current_boundary
        del summary_day

        gc.collect()

    # ========================================================
    # 8. Combine local cluster summaries
    # ========================================================

    if len(cluster_summaries) > 0:

        all_cluster_summaries = pd.concat(
            cluster_summaries,
            ignore_index=True
        )

    else:

        all_cluster_summaries = pd.DataFrame()

    # ========================================================
    # 9. Combine cross-day edges
    # ========================================================

    if len(cross_day_edges) > 0:

        all_cross_day_edges = pd.concat(
            cross_day_edges,
            ignore_index=True
        )

        all_cross_day_edges = (
            all_cross_day_edges
            .drop_duplicates(
                ignore_index=True
            )
        )

    else:

        all_cross_day_edges = pd.DataFrame(
            columns=[
                "local_cluster_id_a",
                "local_cluster_id_b"
            ]
        )

    # ========================================================
    # 10. Final summary
    # ========================================================

    print("\n" + "=" * 70)
    print("TEMPORAL ENGINE COMPLETE")
    print("=" * 70)

    print(
        "Local clusters:",
        f"{len(all_cluster_summaries):,}"
    )

    print(
        "Cross-day cluster edges:",
        f"{len(all_cross_day_edges):,}"
    )

    return (
        all_cluster_summaries,
        all_cross_day_edges
    )


In [54]:
production_clusters_7d, production_edges_7d = (
    process_temporal_range(
        df,
        start_date=pd.Timestamp("2021-03-28").date(),
        end_date=pd.Timestamp("2021-04-03").date()
    )
)


PRODUCTION TEMPORAL ENGINE
Date range: 2021-03-28 → 2021-04-03
Observed dates to process: 7

[1/7] 2021-03-28
2021-03-28: 24,105 detections, 43,813 spatial pairs
2021-03-28: 38,774 valid pairs
2021-03-28: 10,100 local clusters
2021-03-28: 24,105 detections → 10,100 local clusters

[2/7] 2021-03-29
2021-03-29: 27,979 detections, 63,325 spatial pairs
2021-03-29: 54,783 valid pairs
2021-03-29: 10,803 local clusters
2021-03-29: 27,979 detections → 10,803 local clusters
2021-03-28: 0 boundary detections
2021-03-29: 17,859 boundary detections
Cross-day cluster edges: 0

[3/7] 2021-03-30
2021-03-30: 33,403 detections, 78,414 spatial pairs
2021-03-30: 66,716 valid pairs
2021-03-30: 11,967 local clusters
2021-03-30: 33,403 detections → 11,967 local clusters
2021-03-29: 0 boundary detections
2021-03-30: 20,415 boundary detections
Cross-day cluster edges: 0

[4/7] 2021-03-31
2021-03-31: 37,801 detections, 89,322 spatial pairs
2021-03-31: 75,626 valid pairs
2021-03-31: 12,641 local clusters
2021-0

In [55]:
print(
    "Production local clusters:",
    len(production_clusters_7d)
)

print(
    "Production cross-day edges:",
    len(production_edges_7d)
)


Production local clusters: 77244
Production cross-day edges: 0


In [56]:
print(
    "Expected local clusters: 77244"
)

print(
    "Expected cross-day edges: 8738"
)


Expected local clusters: 77244
Expected cross-day edges: 8738


In [59]:
previous_day_df = None
previous_date = None

for current_date in production_dates:

    current_day_df = ...

    # local clustering
    ...

    if previous_day_df is not None:

        edges = cross_day_cluster_edges(
            previous_day_df,
            current_day_df,
            previous_date,
            current_date
        )

        ...

    previous_day_df = current_day_df.copy()
    previous_date = current_date


AttributeError: 'ellipsis' object has no attribute 'copy'

In [57]:
print(
    f"{current_date}: "
    f"boundary type={type(current_boundary)}, "
    f"n={len(current_boundary)}"
)


NameError: name 'current_date' is not defined

In [60]:
import inspect

print(inspect.getsource(process_temporal_range))


def process_temporal_range(
    combined_df,
    start_date,
    end_date,
    verbose=True
):
    """
    Memory-conscious temporal processing.

    For each observed date:
        1. Extract daily detections
        2. Create local clusters
        3. Store local cluster summary
        4. Retain only boundary detections
        5. Match with previous day's boundary
        6. Store cluster-level cross-day edges
        7. Release temporary objects

    Returns
    -------
    all_cluster_summaries : DataFrame
        One row per local cluster.

    all_cross_day_edges : DataFrame
        Unique cross-day local-cluster connections.
    """

    # --------------------------------------------------------
    # Select dates
    # --------------------------------------------------------

    mask = (
        (combined_df["date"] >= start_date)
        &
        (combined_df["date"] <= end_date)
    )

    selected_dates = sorted(
        combined_df.loc[mask, "date"].unique()
    )

    

In [61]:
print(inspect.getsource(cross_day_cluster_edges))


def cross_day_cluster_edges(
    day_a,
    day_b,
    date_a,
    date_b
):
    """
    Find unique local-cluster connections between two dates.

    A cross-day connection requires:
        distance <= 1000 m
        AND
        time difference <= 12 hours

    Only the relevant temporal boundary portions are searched:
        previous day: last 12 hours
        next day:     first 12 hours

    Returns
    -------
    pandas.DataFrame
        Unique local-cluster-to-local-cluster edges.
    """

    # ========================================================
    # 1. Define the temporal boundary
    # ========================================================

    midnight_b = pd.Timestamp(date_b)

    boundary_a_start = (
        midnight_b - pd.Timedelta(hours=12)
    )

    boundary_b_end = (
        midnight_b + pd.Timedelta(hours=12)
    )

    # ========================================================
    # 2. Select only detections that could possibly connect
    # =============

In [62]:
def process_temporal_range(
    combined_df,
    start_date,
    end_date,
    verbose=True
):
    """
    Memory-conscious temporal processing.

    For each observed date:
        1. Extract daily detections
        2. Create local clusters
        3. Store local cluster summary
        4. Match the complete current day against the
           complete previous day using cross_day_cluster_edges()
        5. Store cluster-level cross-day edges
        6. Retain only the current clustered day for the
           next iteration
        7. Release older temporary objects

    Returns
    -------
    all_cluster_summaries : DataFrame
        One row per local cluster.

    all_cross_day_edges : DataFrame
        Unique cross-day local-cluster connections.
    """

    # --------------------------------------------------------
    # 1. Select dates
    # --------------------------------------------------------

    mask = (
        (combined_df["date"] >= start_date)
        &
        (combined_df["date"] <= end_date)
    )

    selected_dates = sorted(
        combined_df.loc[mask, "date"].unique()
    )

    print("=" * 70)
    print("PRODUCTION TEMPORAL ENGINE")
    print("=" * 70)

    print(
        f"Date range: {start_date} → {end_date}"
    )

    print(
        f"Observed dates to process: "
        f"{len(selected_dates):,}"
    )

    # --------------------------------------------------------
    # 2. Storage
    # --------------------------------------------------------

    cluster_summaries = []
    cross_day_edges = []

    # --------------------------------------------------------
    # 3. Previous complete clustered day
    # --------------------------------------------------------

    previous_day_df = None
    previous_date = None

    # --------------------------------------------------------
    # 4. Main temporal loop
    # --------------------------------------------------------

    for counter, current_date in enumerate(
        selected_dates,
        start=1
    ):

        if verbose:
            print(
                f"\n[{counter}/{len(selected_dates)}] "
                f"{current_date}"
            )

        # ====================================================
        # Extract current day
        # ====================================================

        day_mask = (
            combined_df["date"] == current_date
        )

        day_df = combined_df.loc[
            day_mask
        ].copy()

        if len(day_df) == 0:

            print(
                f"{current_date}: no detections"
            )

            continue

        # ====================================================
        # Local clustering
        # ====================================================

        clustered_day, summary_day = (
            cluster_one_day(
                day_df,
                current_date
            )
        )

        if verbose:
            print(
                f"{current_date}: "
                f"{len(day_df):,} detections, "
                f"{len(summary_day):,} local clusters"
            )

        # ====================================================
        # Store local cluster summary
        # ====================================================

        cluster_summaries.append(
            summary_day.copy()
        )

        # ====================================================
        # Cross-day matching
        #
        # IMPORTANT:
        #
        # cross_day_cluster_edges() expects the COMPLETE
        # previous and current daily DataFrames.
        #
        # It internally extracts:
        #   previous day = last 12 hours
        #   current day  = first 12 hours
        # ====================================================

        if (
            previous_day_df is not None
            and
            previous_date is not None
        ):

            # ------------------------------------------------
            # Only connect consecutive calendar dates.
            # ------------------------------------------------

            expected_next_date = (
                previous_date
                +
                pd.Timedelta(days=1)
            )

            if current_date == expected_next_date:

                if verbose:
                    print(
                        f"Comparing previous day "
                        f"{previous_date} → "
                        f"{current_date}"
                    )

                edges_day = cross_day_cluster_edges(
                    previous_day_df,
                    clustered_day,
                    previous_date,
                    current_date
                )

                if len(edges_day) > 0:

                    cross_day_edges.append(
                        edges_day
                    )

                if verbose:
                    print(
                        f"Cross-day cluster edges: "
                        f"{len(edges_day):,}"
                    )

            else:

                if verbose:
                    print(
                        f"No cross-day matching: "
                        f"{previous_date} → "
                        f"{current_date}"
                    )

        # ====================================================
        # Current clustered day becomes previous day
        #
        # IMPORTANT:
        # Keep the COMPLETE clustered day here.
        # Do NOT replace it with a 12-hour boundary.
        # ====================================================

        previous_day_df = clustered_day
        previous_date = current_date

        # ====================================================
        # Release objects that are no longer needed
        # ====================================================

        del day_df
        del summary_day

        gc.collect()

    # ========================================================
    # 5. Combine local cluster summaries
    # ========================================================

    if len(cluster_summaries) > 0:

        all_cluster_summaries = pd.concat(
            cluster_summaries,
            ignore_index=True
        )

    else:

        all_cluster_summaries = pd.DataFrame()

    # ========================================================
    # 6. Combine cross-day edges
    # ========================================================

    if len(cross_day_edges) > 0:

        all_cross_day_edges = pd.concat(
            cross_day_edges,
            ignore_index=True
        )

        all_cross_day_edges = (
            all_cross_day_edges
            .drop_duplicates(
                ignore_index=True
            )
        )

    else:

        all_cross_day_edges = pd.DataFrame(
            columns=[
                "local_cluster_id_a",
                "local_cluster_id_b"
            ]
        )

    # ========================================================
    # 7. Final summary
    # ========================================================

    print("\n" + "=" * 70)
    print("TEMPORAL ENGINE COMPLETE")
    print("=" * 70)

    print(
        "Local clusters:",
        f"{len(all_cluster_summaries):,}"
    )

    print(
        "Cross-day cluster edges:",
        f"{len(all_cross_day_edges):,}"
    )

    return (
        all_cluster_summaries,
        all_cross_day_edges
    )


In [66]:
cluster_summaries_7d, cross_edges_7d = process_temporal_range(
    df,
    pd.Timestamp("2021-03-28").date(),
    pd.Timestamp("2021-04-03").date(),
    verbose=True
)


PRODUCTION TEMPORAL ENGINE
Date range: 2021-03-28 → 2021-04-03
Observed dates to process: 7

[1/7] 2021-03-28
2021-03-28: 24,105 detections, 43,813 spatial pairs
2021-03-28: 38,774 valid pairs
2021-03-28: 10,100 local clusters
2021-03-28: 24,105 detections, 10,100 local clusters

[2/7] 2021-03-29
2021-03-29: 27,979 detections, 63,325 spatial pairs
2021-03-29: 54,783 valid pairs
2021-03-29: 10,803 local clusters
2021-03-29: 27,979 detections, 10,803 local clusters
Comparing previous day 2021-03-28 → 2021-03-29
2021-03-28: 7,883 boundary detections
2021-03-29: 17,859 boundary detections
Cross-day cluster edges: 845

[3/7] 2021-03-30
2021-03-30: 33,403 detections, 78,414 spatial pairs
2021-03-30: 66,716 valid pairs
2021-03-30: 11,967 local clusters
2021-03-30: 33,403 detections, 11,967 local clusters
Comparing previous day 2021-03-29 → 2021-03-30
2021-03-29: 10,120 boundary detections
2021-03-30: 20,415 boundary detections
Cross-day cluster edges: 1,304

[4/7] 2021-03-31
2021-03-31: 37,80

In [67]:
# ============================================================
# SAVE 7-DAY REGRESSION BENCHMARK
# ============================================================

print("7-DAY REGRESSION TEST")
print("=" * 70)

print(f"Local clusters:      {len(cluster_summaries_7d):,}")
print(f"Cross-day edges:     {len(cross_edges_7d):,}")

assert len(cluster_summaries_7d) == 77_244, (
    f"Unexpected local cluster count: "
    f"{len(cluster_summaries_7d):,}"
)

assert len(cross_edges_7d) == 8_738, (
    f"Unexpected cross-day edge count: "
    f"{len(cross_edges_7d):,}"
)

print("\n✅ 7-day regression benchmark PASSED")


7-DAY REGRESSION TEST
Local clusters:      77,244
Cross-day edges:     8,738

✅ 7-day regression benchmark PASSED


In [68]:
print("CLUSTER SUMMARY")
print("=" * 70)

print("Shape:", cluster_summaries_7d.shape)
print("Columns:")
print(cluster_summaries_7d.columns.tolist())

print("\nFirst 5 rows:")
display(cluster_summaries_7d.head())


CLUSTER SUMMARY
Shape: (77244, 15)
Columns:
['local_cluster_id', 'start_time', 'end_time', 'n_detections', 'centroid_x', 'centroid_y', 'total_frp', 'mean_frp', 'max_frp', 'n_observation_times', 'duration_hours', 'satellites', 'n_satellites', 'max_radius_m', 'max_radius_km']

First 5 rows:


,local_cluster_id,start_time,end_time,n_detections,centroid_x,centroid_y,total_frp,mean_frp,max_frp,n_observation_times,duration_hours,satellites,n_satellites,max_radius_m,max_radius_km
0,20210328_0,2021-03-28 06:31:00,2021-03-28 06:31:00,1,8.849165e+06,2.953987e+06,29.70,29.700,29.70,1,0.000000,N20,1,0.000000,0.000000
1,20210328_1,2021-03-28 06:31:00,2021-03-28 06:31:00,1,8.872919e+06,2.960364e+06,4.88,4.880,4.88,1,0.000000,N20,1,0.000000,0.000000
2,20210328_2,2021-03-28 06:31:00,2021-03-28 06:31:00,2,8.863426e+06,2.958056e+06,15.89,7.945,9.15,1,0.000000,N20,1,90.649437,0.090649
3,20210328_3,2021-03-28 06:31:00,2021-03-28 07:21:00,5,8.885147e+06,2.969370e+06,101.38,20.276,22.98,2,0.833333,"N20,SNPP",2,434.603990,0.434604
4,20210328_4,2021-03-28 06:31:00,2021-03-28 06:31:00,1,8.885202e+06,2.975603e+06,7.74,7.740,7.74,1,0.000000,N20,1,0.000000,0.000000


In [69]:
print("\nCROSS-DAY EDGES")
print("=" * 70)

print("Shape:", cross_edges_7d.shape)
print("Columns:")
print(cross_edges_7d.columns.tolist())

print("\nFirst 10 rows:")
display(cross_edges_7d.head(10))



CROSS-DAY EDGES
Shape: (8738, 2)
Columns:
['local_cluster_id_a', 'local_cluster_id_b']

First 10 rows:


,local_cluster_id_a,local_cluster_id_b
0,20210328_4496,20210329_269
1,20210328_117,20210329_275
2,20210328_292,20210329_148
3,20210328_337,20210329_376
4,20210328_4534,20210329_161
5,20210328_4548,20210329_229
6,20210328_4550,20210329_256
7,20210328_218,20210329_179
8,20210328_4558,20210329_193
9,20210328_4560,20210329_290


In [70]:
cluster_ids_7d = set(
    cluster_summaries_7d["local_cluster_id"]
)

edge_a_ids = set(
    cross_edges_7d["local_cluster_id_a"]
)

edge_b_ids = set(
    cross_edges_7d["local_cluster_id_b"]
)

missing_a = edge_a_ids - cluster_ids_7d
missing_b = edge_b_ids - cluster_ids_7d

print("Unique local cluster IDs:", len(cluster_ids_7d))
print("Unique edge A IDs:", len(edge_a_ids))
print("Unique edge B IDs:", len(edge_b_ids))

print("Missing A endpoints:", len(missing_a))
print("Missing B endpoints:", len(missing_b))

assert len(missing_a) == 0
assert len(missing_b) == 0

print("\n✅ All cross-day edge endpoints exist")


Unique local cluster IDs: 77244
Unique edge A IDs: 7722
Unique edge B IDs: 8122
Missing A endpoints: 0
Missing B endpoints: 0

✅ All cross-day edge endpoints exist


In [71]:
import networkx as nx

G_7d = nx.Graph()

# Add every local cluster as a node
G_7d.add_nodes_from(
    cluster_summaries_7d["local_cluster_id"]
)

# Add cross-day connections
G_7d.add_edges_from(
    cross_edges_7d[
        [
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

n_global_components_7d = nx.number_connected_components(G_7d)

print("=" * 70)
print("7-DAY GLOBAL GRAPH")
print("=" * 70)

print("Nodes:", f"{G_7d.number_of_nodes():,}")
print("Edges:", f"{G_7d.number_of_edges():,}")
print(
    "Global components:",
    f"{n_global_components_7d:,}"
)

assert G_7d.number_of_nodes() == 77_244
assert G_7d.number_of_edges() == 8_738
assert n_global_components_7d == 68_514

print("\n✅ 7-day global graph regression PASSED")


7-DAY GLOBAL GRAPH
Nodes: 77,244
Edges: 8,738
Global components: 68,514

✅ 7-day global graph regression PASSED


In [72]:
# ------------------------------------------------------------
# Map each local cluster to its date
# ------------------------------------------------------------

cluster_date_map = (
    cluster_summaries_7d[
        ["local_cluster_id"]
    ]
    .copy()
)

cluster_date_map["cluster_date"] = (
    cluster_date_map["local_cluster_id"]
    .astype(str)
    .str[:8]
)

cluster_date_map = dict(
    zip(
        cluster_date_map["local_cluster_id"],
        cluster_date_map["cluster_date"]
    )
)

# ------------------------------------------------------------
# Determine number of dates represented by each component
# ------------------------------------------------------------

component_spans = []

for component in nx.connected_components(G_7d):

    dates = {
        cluster_date_map[node]
        for node in component
    }

    component_spans.append(len(dates))

component_spans = np.asarray(
    component_spans,
    dtype=np.int16
)

print("=" * 70)
print("MULTI-DAY GLOBAL EVENT VALIDATION")
print("=" * 70)

print(
    "Global events:",
    f"{len(component_spans):,}"
)

print(
    "Events spanning >=2 days:",
    f"{(component_spans >= 2).sum():,}"
)

print(
    "Events spanning >=3 days:",
    f"{(component_spans >= 3).sum():,}"
)

print(
    "Events spanning >=4 days:",
    f"{(component_spans >= 4).sum():,}"
)

print(
    "Events spanning >=5 days:",
    f"{(component_spans >= 5).sum():,}"
)

print(
    "Maximum days:",
    f"{component_spans.max()}"
)


MULTI-DAY GLOBAL EVENT VALIDATION
Global events: 68,514
Events spanning >=2 days: 5,999
Events spanning >=3 days: 785
Events spanning >=4 days: 199
Events spanning >=5 days: 48
Maximum days: 7


In [74]:
# ============================================================
# ONE-MONTH PRODUCTION TEST
# ============================================================

month_start = pd.Timestamp("2021-03-01").date()
month_end   = pd.Timestamp("2021-03-31").date()

cluster_summaries_month, cross_edges_month = (
    process_temporal_range(
        df,
        month_start,
        month_end,
        verbose=True
    )
)

print("\n" + "=" * 70)
print("ONE-MONTH TEST COMPLETE")
print("=" * 70)

print(
    "Local clusters:",
    f"{len(cluster_summaries_month):,}"
)

print(
    "Cross-day edges:",
    f"{len(cross_edges_month):,}"
)


PRODUCTION TEMPORAL ENGINE
Date range: 2021-03-01 → 2021-03-31
Observed dates to process: 31

[1/31] 2021-03-01
2021-03-01: 12,085 detections, 21,565 spatial pairs
2021-03-01: 19,030 valid pairs
2021-03-01: 5,229 local clusters
2021-03-01: 12,085 detections, 5,229 local clusters

[2/31] 2021-03-02
2021-03-02: 12,975 detections, 25,015 spatial pairs
2021-03-02: 22,322 valid pairs
2021-03-02: 5,258 local clusters
2021-03-02: 12,975 detections, 5,258 local clusters
Comparing previous day 2021-03-01 → 2021-03-02
2021-03-01: 4,388 boundary detections
2021-03-02: 7,525 boundary detections
Cross-day cluster edges: 432

[3/31] 2021-03-03
2021-03-03: 15,954 detections, 32,093 spatial pairs
2021-03-03: 27,901 valid pairs
2021-03-03: 6,356 local clusters
2021-03-03: 15,954 detections, 6,356 local clusters
Comparing previous day 2021-03-02 → 2021-03-03
2021-03-02: 5,450 boundary detections
2021-03-03: 8,844 boundary detections
Cross-day cluster edges: 611

[4/31] 2021-03-04
2021-03-04: 19,025 dete

In [75]:
print("Cluster summary shape:", cluster_summaries_month.shape)
print("Cross-edge shape:", cross_edges_month.shape)

print("\nCluster columns:")
print(cluster_summaries_month.columns.tolist())

print("\nEdge columns:")
print(cross_edges_month.columns.tolist())

print("\nNull counts in cluster summary:")
print(
    cluster_summaries_month.isna().sum()
)

print("\nDuplicate local cluster IDs:")
print(
    cluster_summaries_month["local_cluster_id"].duplicated().sum()
)


Cluster summary shape: (230239, 15)
Cross-edge shape: (21379, 2)

Cluster columns:
['local_cluster_id', 'start_time', 'end_time', 'n_detections', 'centroid_x', 'centroid_y', 'total_frp', 'mean_frp', 'max_frp', 'n_observation_times', 'duration_hours', 'satellites', 'n_satellites', 'max_radius_m', 'max_radius_km']

Edge columns:
['local_cluster_id_a', 'local_cluster_id_b']

Null counts in cluster summary:
local_cluster_id       0
start_time             0
end_time               0
n_detections           0
centroid_x             0
centroid_y             0
total_frp              0
mean_frp               0
max_frp                0
n_observation_times    0
duration_hours         0
satellites             0
n_satellites           0
max_radius_m           0
max_radius_km          0
dtype: int64

Duplicate local cluster IDs:
0


In [76]:
G_month = nx.Graph()

G_month.add_nodes_from(
    cluster_summaries_month["local_cluster_id"]
)

G_month.add_edges_from(
    cross_edges_month[
        [
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

print("=" * 70)
print("MARCH 2021 GLOBAL GRAPH")
print("=" * 70)

print("Nodes:", f"{G_month.number_of_nodes():,}")
print("Edges:", f"{G_month.number_of_edges():,}")
print(
    "Global components:",
    f"{nx.number_connected_components(G_month):,}"
)

assert (
    G_month.number_of_nodes()
    ==
    len(cluster_summaries_month)
)

assert (
    G_month.number_of_edges()
    ==
    len(cross_edges_month)
)

print("\n✅ Month-level graph construction passed")


MARCH 2021 GLOBAL GRAPH
Nodes: 230,239
Edges: 21,379
Global components: 208,884

✅ Month-level graph construction passed


In [77]:
import os
import psutil

process = psutil.Process(os.getpid())

memory_gb = (
    process.memory_info().rss
    / (1024 ** 3)
)

print(
    f"Current Python process memory: "
    f"{memory_gb:.2f} GB"
)


Current Python process memory: 8.21 GB


In [78]:
from pathlib import Path

PRODUCTION_DIR = Path("fire_event_production")

CLUSTER_DIR = PRODUCTION_DIR / "cluster_summaries"
EDGE_DIR = PRODUCTION_DIR / "cross_day_edges"

CLUSTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EDGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Production directory:", PRODUCTION_DIR.resolve())
print("Cluster directory:", CLUSTER_DIR.resolve())
print("Edge directory:", EDGE_DIR.resolve())


Production directory: /content/fire_event_production
Cluster directory: /content/fire_event_production/cluster_summaries
Edge directory: /content/fire_event_production/cross_day_edges


In [79]:
cluster_summaries_month.to_parquet(
    CLUSTER_DIR / "clusters_2021-03.parquet",
    index=False
)

cross_edges_month.to_parquet(
    EDGE_DIR / "edges_2021-03.parquet",
    index=False
)

print("March cluster file saved.")
print("March edge file saved.")


March cluster file saved.
March edge file saved.


In [80]:
test_clusters = pd.read_parquet(
    CLUSTER_DIR / "clusters_2021-03.parquet"
)

test_edges = pd.read_parquet(
    EDGE_DIR / "edges_2021-03.parquet"
)

print(
    "Reloaded clusters:",
    test_clusters.shape
)

print(
    "Reloaded edges:",
    test_edges.shape
)

assert len(test_clusters) == 230_239
assert len(test_edges) == 21_379

print("\n✅ Disk write/read validation passed")


Reloaded clusters: (230239, 15)
Reloaded edges: (21379, 2)

✅ Disk write/read validation passed


In [90]:
def process_temporal_range_disk(
    combined_df,
    start_date,
    end_date,
    output_dir,
    verbose=True
):
    """
    Disk-backed production temporal engine.

    Daily cluster summaries and cross-day edges are written
    directly to Parquet files.
    """

    output_dir = Path(output_dir)

    cluster_dir = (
        output_dir / "cluster_summaries"
    )

    edge_dir = (
        output_dir / "cross_day_edges"
    )

    cluster_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    edge_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Select dates
    # --------------------------------------------------------

    mask = (
        (combined_df["date"] >= start_date)
        &
        (combined_df["date"] <= end_date)
    )

    selected_dates = sorted(
        combined_df.loc[
            mask,
            "date"
        ].unique()
    )

    print("=" * 70)
    print("DISK-BACKED TEMPORAL ENGINE")
    print("=" * 70)

    print(
        f"Date range: "
        f"{start_date} → {end_date}"
    )

    print(
        f"Observed dates: "
        f"{len(selected_dates):,}"
    )

    cluster_files = []
    edge_files = []

    previous_day_df = None
    previous_date = None

    # --------------------------------------------------------
    # Main loop
    # --------------------------------------------------------

    for counter, current_date in enumerate(
        selected_dates,
        start=1
    ):

        print(
            f"\n[{counter}/{len(selected_dates)}] "
            f"{current_date}"
        )

        # IMPORTANT: reset every iteration
        edges_day = None

        # ====================================================
        # Extract current day
        # ====================================================

        day_df = combined_df[
            combined_df["date"] == current_date
        ].copy()

        if len(day_df) == 0:

            print(
                f"{current_date}: "
                f"no detections"
            )

            continue

        # ====================================================
        # Local clustering
        # ====================================================

        clustered_day, summary_day = (
            cluster_one_day(
                day_df,
                current_date
            )
        )

        print(
            f"{current_date}: "
            f"{len(day_df):,} detections, "
            f"{len(summary_day):,} local clusters"
        )

        # ====================================================
        # Save cluster summary
        # ====================================================

        cluster_file = (
            cluster_dir
            /
            f"clusters_{current_date}.parquet"
        )

        summary_day.to_parquet(
            cluster_file,
            index=False
        )

        cluster_files.append(
            cluster_file
        )

        # ====================================================
        # Cross-day matching
        # ====================================================

        if (
            previous_day_df is not None
            and
            previous_date is not None
        ):

            expected_next_date = (
                previous_date
                +
                pd.Timedelta(days=1)
            )

            if current_date == expected_next_date:

                print(
                    f"Comparing previous day "
                    f"{previous_date} → "
                    f"{current_date}"
                )

                edges_day = (
                    cross_day_cluster_edges(
                        previous_day_df,
                        clustered_day,
                        previous_date,
                        current_date
                    )
                )

                print(
                    f"Cross-day cluster edges: "
                    f"{len(edges_day):,}"
                )

                # --------------------------------------------
                # SAVE EDGE FILE
                # --------------------------------------------

                if len(edges_day) > 0:

                    edge_file = (
                        edge_dir
                        /
                        f"edges_"
                        f"{previous_date}_"
                        f"{current_date}.parquet"
                    )

                    edges_day.to_parquet(
                        edge_file,
                        index=False
                    )

                    edge_files.append(
                        edge_file
                    )

                    print(
                        f"Saved edge file: "
                        f"{edge_file.name}"
                    )

            else:

                print(
                    f"No cross-day matching: "
                    f"{previous_date} → "
                    f"{current_date}"
                )

        # ====================================================
        # Previous day becomes current day
        # ====================================================

        previous_day_df = clustered_day
        previous_date = current_date

        # ====================================================
        # Release memory
        # ====================================================

        del day_df
        del clustered_day
        del summary_day

        if edges_day is not None:
            del edges_day

        gc.collect()

    # ========================================================
    # Final report
    # ========================================================

    print("\n" + "=" * 70)
    print("DISK-BACKED ENGINE COMPLETE")
    print("=" * 70)

    print(
        "Cluster files:",
        f"{len(cluster_files):,}"
    )

    print(
        "Edge files:",
        f"{len(edge_files):,}"
    )

    return (
        cluster_files,
        edge_files
    )


In [83]:
disk_test_dir = (
    Path("fire_event_production")
    / "disk_test"
)

cluster_files_test, edge_files_test = (
    process_temporal_range_disk(
        df,
        pd.Timestamp("2021-03-01").date(),
        pd.Timestamp("2021-03-31").date(),
        disk_test_dir,
        verbose=True
    )
)


DISK-BACKED TEMPORAL ENGINE
Date range: 2021-03-01 → 2021-03-31
Observed dates: 31

[1/31] 2021-03-01
2021-03-01: 12,085 detections, 21,565 spatial pairs
2021-03-01: 19,030 valid pairs
2021-03-01: 5,229 local clusters
2021-03-01: 12,085 detections, 5,229 local clusters

[2/31] 2021-03-02
2021-03-02: 12,975 detections, 25,015 spatial pairs
2021-03-02: 22,322 valid pairs
2021-03-02: 5,258 local clusters
2021-03-02: 12,975 detections, 5,258 local clusters
2021-03-01: 0 boundary detections
2021-03-02: 7,525 boundary detections
Cross-day cluster edges: 0

[3/31] 2021-03-03
2021-03-03: 15,954 detections, 32,093 spatial pairs
2021-03-03: 27,901 valid pairs
2021-03-03: 6,356 local clusters
2021-03-03: 15,954 detections, 6,356 local clusters
2021-03-02: 0 boundary detections
2021-03-03: 8,844 boundary detections
Cross-day cluster edges: 0

[4/31] 2021-03-04
2021-03-04: 19,025 detections, 38,064 spatial pairs
2021-03-04: 33,033 valid pairs
2021-03-04: 7,258 local clusters
2021-03-04: 19,025 dete

In [85]:
# ------------------------------------------------------------
# Read all saved cluster files
# ------------------------------------------------------------

cluster_parts = [
    pd.read_parquet(f)
    for f in cluster_files_test
]

clusters_disk_test = pd.concat(
    cluster_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Read all saved edge files
# ------------------------------------------------------------

if len(edge_files_test) > 0:

    edge_parts = [
        pd.read_parquet(f)
        for f in edge_files_test
    ]

    edges_disk_test = pd.concat(
        edge_parts,
        ignore_index=True
    )

else:

    edges_disk_test = pd.DataFrame(
        columns=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("=" * 70)
print("DISK-BACKED MARCH VALIDATION")
print("=" * 70)

print(
    "Local clusters:",
    f"{len(clusters_disk_test):,}"
)

print(
    "Cross-day edges:",
    f"{len(edges_disk_test):,}"
)

assert len(clusters_disk_test) == 230_239
assert len(edges_disk_test) == 21_379

assert (
    clusters_disk_test[
        "local_cluster_id"
    ].duplicated().sum()
    ==
    0
)

print("\n✅ Disk-backed March test PASSED")


DISK-BACKED MARCH VALIDATION
Local clusters: 230,239
Cross-day edges: 0


AssertionError: 

In [87]:
# ============================================================
# DIRECT CROSS-DAY EDGE TEST
# ============================================================

date_a = pd.Timestamp("2021-03-01").date()
date_b = pd.Timestamp("2021-03-02").date()

day_a = df[
    df["date"] == date_a
].copy()

day_b = df[
    df["date"] == date_b
].copy()

clustered_a, summary_a = cluster_one_day(
    day_a,
    date_a
)

clustered_b, summary_b = cluster_one_day(
    day_b,
    date_b
)

midnight_b = pd.Timestamp(date_b)

boundary_a = clustered_a[
    clustered_a["datetime"] >= (
        midnight_b - pd.Timedelta(hours=12)
    )
].copy()

boundary_b = clustered_b[
    clustered_b["datetime"] < (
        midnight_b + pd.Timedelta(hours=12)
    )
].copy()

print("=" * 70)
print("DIRECT CROSS-DAY TEST")
print("=" * 70)

print("Day A detections:", len(day_a))
print("Day B detections:", len(day_b))
print("A boundary:", len(boundary_a))
print("B boundary:", len(boundary_b))

edges_test = cross_day_cluster_edges(
    clustered_a,
    clustered_b,
    date_a,
    date_b
)

print(
    "Cross-day edges:",
    len(edges_test)
)

display(edges_test.head())


2021-03-01: 12,085 detections, 21,565 spatial pairs
2021-03-01: 19,030 valid pairs
2021-03-01: 5,229 local clusters
2021-03-02: 12,975 detections, 25,015 spatial pairs
2021-03-02: 22,322 valid pairs
2021-03-02: 5,258 local clusters
DIRECT CROSS-DAY TEST
Day A detections: 12085
Day B detections: 12975
A boundary: 4388
B boundary: 7525
2021-03-01: 4,388 boundary detections
2021-03-02: 7,525 boundary detections
Cross-day edges: 432


,local_cluster_id_a,local_cluster_id_b
0,20210301_1988,20210302_224
1,20210301_1996,20210302_202
2,20210301_2023,20210302_13
3,20210301_2032,20210302_171
4,20210301_2037,20210302_186


In [88]:
# ============================================================
# MANUAL EDGE FILE WRITE TEST
# ============================================================

manual_edge_dir = (
    Path("fire_event_production")
    / "manual_edge_test"
)

manual_edge_dir.mkdir(
    parents=True,
    exist_ok=True
)

manual_edge_file = (
    manual_edge_dir
    / "edges_2021-03-01_2021-03-02.parquet"
)

edges_test.to_parquet(
    manual_edge_file,
    index=False
)

print("Saved to:")
print(manual_edge_file.resolve())

print(
    "File exists:",
    manual_edge_file.exists()
)

reloaded_edges = pd.read_parquet(
    manual_edge_file
)

print(
    "Reloaded edges:",
    len(reloaded_edges)
)

assert len(reloaded_edges) == len(edges_test)

print("\n✅ Manual edge write/read passed")


Saved to:
/content/fire_event_production/manual_edge_test/edges_2021-03-01_2021-03-02.parquet
File exists: True
Reloaded edges: 432

✅ Manual edge write/read passed


In [89]:
# ============================================================
# INSPECT DISK-BACKED OUTPUT
# ============================================================

print("=" * 70)
print("DISK OUTPUT INSPECTION")
print("=" * 70)

print("\nCluster files:")

for f in sorted(
    Path(disk_test_dir)
    .joinpath("cluster_summaries")
    .glob("*.parquet")
)[:5]:

    print(f.name)

print("\nEdge directory contents:")

edge_files_on_disk = list(
    Path(disk_test_dir)
    .joinpath("cross_day_edges")
    .glob("*")
)

print(
    "Number of files:",
    len(edge_files_on_disk)
)

for f in sorted(edge_files_on_disk)[:10]:
    print(f.name)


DISK OUTPUT INSPECTION

Cluster files:
clusters_2021-03-01.parquet
clusters_2021-03-02.parquet
clusters_2021-03-03.parquet
clusters_2021-03-04.parquet
clusters_2021-03-05.parquet

Edge directory contents:
Number of files: 0


In [92]:
# ============================================================
# TWO-DAY DISK TEST
# ============================================================

two_day_test_dir = (
    Path("fire_event_production")
    / "two_day_test"
)

cluster_files_2d, edge_files_2d = (
    process_temporal_range_disk(
        df,
        pd.Timestamp("2021-03-01").date(),
        pd.Timestamp("2021-03-02").date(),
        two_day_test_dir,
        verbose=True
    )
)

print("\n" + "=" * 70)
print("TWO-DAY DISK TEST")
print("=" * 70)

print(
    "Cluster files:",
    len(cluster_files_2d)
)

print(
    "Edge files:",
    len(edge_files_2d)
)


DISK-BACKED TEMPORAL ENGINE
Date range: 2021-03-01 → 2021-03-02
Observed dates: 2

[1/2] 2021-03-01
2021-03-01: 12,085 detections, 21,565 spatial pairs
2021-03-01: 19,030 valid pairs
2021-03-01: 5,229 local clusters
2021-03-01: 12,085 detections, 5,229 local clusters

[2/2] 2021-03-02
2021-03-02: 12,975 detections, 25,015 spatial pairs
2021-03-02: 22,322 valid pairs
2021-03-02: 5,258 local clusters
2021-03-02: 12,975 detections, 5,258 local clusters
Comparing previous day 2021-03-01 → 2021-03-02
2021-03-01: 4,388 boundary detections
2021-03-02: 7,525 boundary detections
Cross-day cluster edges: 432
Saved edge file: edges_2021-03-01_2021-03-02.parquet

DISK-BACKED ENGINE COMPLETE
Cluster files: 2
Edge files: 1

TWO-DAY DISK TEST
Cluster files: 2
Edge files: 1


In [93]:
edges_2d_disk = pd.read_parquet(
    edge_files_2d[0]
)

print(
    "Saved edge rows:",
    len(edges_2d_disk)
)

display(edges_2d_disk.head())


Saved edge rows: 432


,local_cluster_id_a,local_cluster_id_b
0,20210301_1988,20210302_224
1,20210301_1996,20210302_202
2,20210301_2023,20210302_13
3,20210301_2032,20210302_171
4,20210301_2037,20210302_186


In [95]:
# ============================================================
# FRESH MARCH DISK-BACKED PRODUCTION TEST
# ============================================================

march_disk_dir = (
    Path("fire_event_production")
    / "march_2021_final_test"
)

# Create clean directories
if march_disk_dir.exists():
    import shutil
    shutil.rmtree(march_disk_dir)

march_disk_dir.mkdir(
    parents=True,
    exist_ok=True
)

cluster_files_march, edge_files_march = (
    process_temporal_range_disk(
        df,
        pd.Timestamp("2021-03-01").date(),
        pd.Timestamp("2021-03-31").date(),
        march_disk_dir,
        verbose=True
    )
)

print("\n" + "=" * 70)
print("MARCH DISK-BACKED TEST COMPLETE")
print("=" * 70)

print(
    "Cluster files:",
    len(cluster_files_march)
)

print(
    "Edge files:",
    len(edge_files_march)
)


DISK-BACKED TEMPORAL ENGINE
Date range: 2021-03-01 → 2021-03-31
Observed dates: 31

[1/31] 2021-03-01
2021-03-01: 12,085 detections, 21,565 spatial pairs
2021-03-01: 19,030 valid pairs
2021-03-01: 5,229 local clusters
2021-03-01: 12,085 detections, 5,229 local clusters

[2/31] 2021-03-02
2021-03-02: 12,975 detections, 25,015 spatial pairs
2021-03-02: 22,322 valid pairs
2021-03-02: 5,258 local clusters
2021-03-02: 12,975 detections, 5,258 local clusters
Comparing previous day 2021-03-01 → 2021-03-02
2021-03-01: 4,388 boundary detections
2021-03-02: 7,525 boundary detections
Cross-day cluster edges: 432
Saved edge file: edges_2021-03-01_2021-03-02.parquet

[3/31] 2021-03-03
2021-03-03: 15,954 detections, 32,093 spatial pairs
2021-03-03: 27,901 valid pairs
2021-03-03: 6,356 local clusters
2021-03-03: 15,954 detections, 6,356 local clusters
Comparing previous day 2021-03-02 → 2021-03-03
2021-03-02: 5,450 boundary detections
2021-03-03: 8,844 boundary detections
Cross-day cluster edges: 611

In [96]:
# ============================================================
# LOAD ALL MARCH DISK RESULTS
# ============================================================

cluster_parts = [
    pd.read_parquet(f)
    for f in cluster_files_march
]

clusters_march_disk = pd.concat(
    cluster_parts,
    ignore_index=True
)

if len(edge_files_march) > 0:

    edge_parts = [
        pd.read_parquet(f)
        for f in edge_files_march
    ]

    edges_march_disk = pd.concat(
        edge_parts,
        ignore_index=True
    )

else:

    edges_march_disk = pd.DataFrame(
        columns=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )

print("=" * 70)
print("MARCH DISK RESULT")
print("=" * 70)

print(
    "Local clusters:",
    f"{len(clusters_march_disk):,}"
)

print(
    "Cross-day edges:",
    f"{len(edges_march_disk):,}"
)

print(
    "Cluster files:",
    len(cluster_files_march)
)

print(
    "Edge files:",
    len(edge_files_march)
)


MARCH DISK RESULT
Local clusters: 230,239
Cross-day edges: 21,379
Cluster files: 31
Edge files: 30


In [97]:
# ============================================================
# MARCH DISK-BACKED REGRESSION ASSERTIONS
# ============================================================

assert len(clusters_march_disk) == 230_239, (
    f"Expected 230,239 clusters, "
    f"got {len(clusters_march_disk):,}"
)

assert len(edges_march_disk) == 21_379, (
    f"Expected 21,379 edges, "
    f"got {len(edges_march_disk):,}"
)

assert (
    clusters_march_disk["local_cluster_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    edges_march_disk[
        "local_cluster_id_a"
    ].isna().sum()
    == 0
)

assert (
    edges_march_disk[
        "local_cluster_id_b"
    ].isna().sum()
    == 0
)

print("=" * 70)
print("✅ MARCH DISK-BACKED REGRESSION PASSED")
print("=" * 70)

print("Local clusters: 230,239")
print("Cross-day edges: 21,379")


✅ MARCH DISK-BACKED REGRESSION PASSED
Local clusters: 230,239
Cross-day edges: 21,379


In [98]:
# ============================================================
# EDGE UNIQUENESS CHECK
# ============================================================

edge_duplicates = (
    edges_march_disk
    .duplicated(
        subset=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )
    .sum()
)

print(
    "Duplicate cross-day edges:",
    edge_duplicates
)

assert edge_duplicates == 0

print("\n✅ Cross-day edge uniqueness passed")


Duplicate cross-day edges: 0

✅ Cross-day edge uniqueness passed


In [99]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

# ============================================================
# BUILD GLOBAL GRAPH FROM DISK-BACKED MARCH RESULTS
# ============================================================

print("=" * 70)
print("MARCH DISK-BACKED GLOBAL GRAPH")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load cluster summaries
# ------------------------------------------------------------

cluster_parts = [
    pd.read_parquet(f)
    for f in sorted(cluster_files_march)
]

clusters_march = pd.concat(
    cluster_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 2. Load cross-day edges
# ------------------------------------------------------------

edge_parts = [
    pd.read_parquet(f)
    for f in sorted(edge_files_march)
]

edges_march = pd.concat(
    edge_parts,
    ignore_index=True
)

print(
    f"Nodes from cluster files: "
    f"{len(clusters_march):,}"
)

print(
    f"Edges from edge files: "
    f"{len(edges_march):,}"
)

# ------------------------------------------------------------
# 3. Extract node IDs
# ------------------------------------------------------------

node_ids = (
    clusters_march[
        "local_cluster_id"
    ]
    .astype(str)
    .to_numpy()
)

# ------------------------------------------------------------
# 4. Validate edge endpoints
# ------------------------------------------------------------

node_set = set(node_ids)

missing_a = (
    ~edges_march[
        "local_cluster_id_a"
    ]
    .astype(str)
    .isin(node_set)
).sum()

missing_b = (
    ~edges_march[
        "local_cluster_id_b"
    ]
    .astype(str)
    .isin(node_set)
).sum()

print(
    f"Missing A endpoints: {missing_a}"
)

print(
    f"Missing B endpoints: {missing_b}"
)

assert missing_a == 0
assert missing_b == 0

# ------------------------------------------------------------
# 5. Build graph
# ------------------------------------------------------------

G_march = nx.Graph()

G_march.add_nodes_from(
    node_ids
)

G_march.add_edges_from(
    edges_march[
        [
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    ]
    .astype(str)
    .itertuples(
        index=False,
        name=None
    )
)

# ------------------------------------------------------------
# 6. Connected components
# ------------------------------------------------------------

n_components_march = nx.number_connected_components(
    G_march
)

print("\n" + "=" * 70)
print("MARCH GLOBAL GRAPH COMPLETE")
print("=" * 70)

print(
    f"Nodes: {G_march.number_of_nodes():,}"
)

print(
    f"Edges: {G_march.number_of_edges():,}"
)

print(
    f"Global components: "
    f"{n_components_march:,}"
)


MARCH DISK-BACKED GLOBAL GRAPH
Nodes from cluster files: 230,239
Edges from edge files: 21,379
Missing A endpoints: 0
Missing B endpoints: 0

MARCH GLOBAL GRAPH COMPLETE
Nodes: 230,239
Edges: 21,379
Global components: 208,884


In [100]:
# ============================================================
# ASSIGN GLOBAL FIRE EVENT IDs
# ============================================================

print("=" * 70)
print("ASSIGNING GLOBAL FIRE EVENT IDs")
print("=" * 70)

# ------------------------------------------------------------
# Create deterministic component labels
# ------------------------------------------------------------

components = nx.connected_components(G_march)

component_records = []

for component_number, nodes in enumerate(
    components,
    start=1
):

    for local_cluster_id in nodes:

        component_records.append(
            (
                local_cluster_id,
                f"FE202103_{component_number:06d}"
            )
        )

cluster_to_event = pd.DataFrame(
    component_records,
    columns=[
        "local_cluster_id",
        "fire_event_id"
    ]
)

print(
    f"Cluster → event mappings: "
    f"{len(cluster_to_event):,}"
)

print(
    f"Unique fire events: "
    f"{cluster_to_event['fire_event_id'].nunique():,}"
)

# ------------------------------------------------------------
# Merge event IDs into cluster summary
# ------------------------------------------------------------

clusters_march_events = clusters_march.merge(
    cluster_to_event,
    on="local_cluster_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(
    clusters_march_events
) == len(clusters_march)

assert (
    clusters_march_events[
        "fire_event_id"
    ].isna().sum()
    == 0
)

assert (
    clusters_march_events[
        "local_cluster_id"
    ].nunique()
    ==
    clusters_march_events[
        "fire_event_id"
    ].nunique()
    # This assertion is intentionally NOT used:
    # multiple local clusters can belong to one global event.
)

print("\n✅ Every local cluster received a global event ID")


ASSIGNING GLOBAL FIRE EVENT IDs
Cluster → event mappings: 230,239
Unique fire events: 208,884


AssertionError: 

In [101]:
# ============================================================
# GLOBAL EVENT ID VALIDATION
# ============================================================

print("=" * 70)
print("GLOBAL EVENT ID VALIDATION")
print("=" * 70)

n_local_clusters = (
    clusters_march_events[
        "local_cluster_id"
    ].nunique()
)

n_global_events = (
    clusters_march_events[
        "fire_event_id"
    ].nunique()
)

print(
    f"Local clusters: {n_local_clusters:,}"
)

print(
    f"Global fire events: {n_global_events:,}"
)

assert n_local_clusters == 230_239
assert n_global_events == 208_884

print("\n✅ Global event-ID validation passed")


GLOBAL EVENT ID VALIDATION
Local clusters: 230,239
Global fire events: 208,884

✅ Global event-ID validation passed


In [102]:
# ============================================================
# LOCAL CLUSTERS PER GLOBAL EVENT
# ============================================================

clusters_per_event = (
    clusters_march_events
    .groupby("fire_event_id")
    .size()
)

print("=" * 70)
print("GLOBAL EVENT STRUCTURE")
print("=" * 70)

print(
    f"Global events: "
    f"{len(clusters_per_event):,}"
)

print(
    f"Events containing >1 local cluster: "
    f"{(clusters_per_event > 1).sum():,}"
)

print(
    f"Maximum local clusters in one event: "
    f"{clusters_per_event.max():,}"
)

print(
    "\nDistribution:"
)

print(
    clusters_per_event
    .describe()
)


GLOBAL EVENT STRUCTURE
Global events: 208,884
Events containing >1 local cluster: 15,334
Maximum local clusters in one event: 39

Distribution:
count    208884.000000
mean          1.102234
std           0.485500
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          39.000000
dtype: float64


In [103]:
from pathlib import Path
import shutil

# ============================================================
# TWO-MONTH DISK-BACKED PRODUCTION TEST
# ============================================================

two_month_dir = (
    Path("fire_event_production")
    / "2021_mar_apr_test"
)

if two_month_dir.exists():
    shutil.rmtree(two_month_dir)

two_month_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(two_month_dir)


Output directory:
fire_event_production/2021_mar_apr_test


In [105]:
# ============================================================
# RUN MARCH + APRIL 2021
# ============================================================

clusters_2m_files, edges_2m_files = (
    process_temporal_range_disk(
        df,
        pd.Timestamp("2021-03-01").date(),
        pd.Timestamp("2021-04-30").date(),
        two_month_dir,
        verbose=True
    )
)

print("\n" + "=" * 70)
print("TWO-MONTH DISK ENGINE COMPLETE")
print("=" * 70)

print(
    "Cluster files:",
    len(clusters_2m_files)
)

print(
    "Edge files:",
    len(edges_2m_files)
)


DISK-BACKED TEMPORAL ENGINE
Date range: 2021-03-01 → 2021-04-30
Observed dates: 61

[1/61] 2021-03-01
2021-03-01: 12,085 detections, 21,565 spatial pairs
2021-03-01: 19,030 valid pairs
2021-03-01: 5,229 local clusters
2021-03-01: 12,085 detections, 5,229 local clusters

[2/61] 2021-03-02
2021-03-02: 12,975 detections, 25,015 spatial pairs
2021-03-02: 22,322 valid pairs
2021-03-02: 5,258 local clusters
2021-03-02: 12,975 detections, 5,258 local clusters
Comparing previous day 2021-03-01 → 2021-03-02
2021-03-01: 4,388 boundary detections
2021-03-02: 7,525 boundary detections
Cross-day cluster edges: 432
Saved edge file: edges_2021-03-01_2021-03-02.parquet

[3/61] 2021-03-03
2021-03-03: 15,954 detections, 32,093 spatial pairs
2021-03-03: 27,901 valid pairs
2021-03-03: 6,356 local clusters
2021-03-03: 15,954 detections, 6,356 local clusters
Comparing previous day 2021-03-02 → 2021-03-03
2021-03-02: 5,450 boundary detections
2021-03-03: 8,844 boundary detections
Cross-day cluster edges: 611

In [106]:
# ============================================================
# LOAD TWO-MONTH DISK RESULTS
# ============================================================

cluster_parts_2m = [
    pd.read_parquet(f)
    for f in sorted(clusters_2m_files)
]

clusters_2m = pd.concat(
    cluster_parts_2m,
    ignore_index=True
)

if len(edges_2m_files) > 0:

    edge_parts_2m = [
        pd.read_parquet(f)
        for f in sorted(edges_2m_files)
    ]

    edges_2m = pd.concat(
        edge_parts_2m,
        ignore_index=True
    )

else:

    edges_2m = pd.DataFrame(
        columns=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )

print("=" * 70)
print("TWO-MONTH DISK RESULTS")
print("=" * 70)

print(
    f"Local clusters: {len(clusters_2m):,}"
)

print(
    f"Cross-day edges: {len(edges_2m):,}"
)

print(
    f"Cluster files: {len(clusters_2m_files)}"
)

print(
    f"Edge files: {len(edges_2m_files)}"
)


TWO-MONTH DISK RESULTS
Local clusters: 366,067
Cross-day edges: 33,490
Cluster files: 61
Edge files: 60


In [107]:
# ============================================================
# CLUSTER ID VALIDATION
# ============================================================

duplicate_clusters_2m = (
    clusters_2m[
        "local_cluster_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate local cluster IDs:",
    duplicate_clusters_2m
)

assert duplicate_clusters_2m == 0

assert (
    clusters_2m["local_cluster_id"]
    .isna()
    .sum()
    == 0
)

print(
    "\n✅ Two-month cluster validation passed"
)


Duplicate local cluster IDs: 0

✅ Two-month cluster validation passed


In [108]:
# ============================================================
# EDGE ENDPOINT VALIDATION
# ============================================================

node_set_2m = set(
    clusters_2m[
        "local_cluster_id"
    ].astype(str)
)

missing_a_2m = (
    ~edges_2m[
        "local_cluster_id_a"
    ]
    .astype(str)
    .isin(node_set_2m)
).sum()

missing_b_2m = (
    ~edges_2m[
        "local_cluster_id_b"
    ]
    .astype(str)
    .isin(node_set_2m)
).sum()

print(
    "Missing A endpoints:",
    missing_a_2m
)

print(
    "Missing B endpoints:",
    missing_b_2m
)

assert missing_a_2m == 0
assert missing_b_2m == 0

print(
    "\n✅ All cross-day edge endpoints are valid"
)


Missing A endpoints: 0
Missing B endpoints: 0

✅ All cross-day edge endpoints are valid


In [109]:
# ============================================================
# EDGE UNIQUENESS
# ============================================================

duplicate_edges_2m = (
    edges_2m
    .duplicated(
        subset=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )
    .sum()
)

print(
    "Duplicate cross-day edges:",
    duplicate_edges_2m
)

assert duplicate_edges_2m == 0

print(
    "\n✅ Two-month edge uniqueness passed"
)


Duplicate cross-day edges: 0

✅ Two-month edge uniqueness passed


In [110]:
# ============================================================
# TWO-MONTH GLOBAL GRAPH
# ============================================================

import networkx as nx

G_2m = nx.Graph()

# Add every local cluster as a node
G_2m.add_nodes_from(
    clusters_2m[
        "local_cluster_id"
    ]
    .astype(str)
)

# Add cross-day connections
G_2m.add_edges_from(
    edges_2m[
        [
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    ]
    .astype(str)
    .itertuples(
        index=False,
        name=None
    )
)

n_components_2m = (
    nx.number_connected_components(G_2m)
)

print("=" * 70)
print("TWO-MONTH GLOBAL GRAPH")
print("=" * 70)

print(
    f"Nodes: {G_2m.number_of_nodes():,}"
)

print(
    f"Edges: {G_2m.number_of_edges():,}"
)

print(
    f"Global components: "
    f"{n_components_2m:,}"
)


TWO-MONTH GLOBAL GRAPH
Nodes: 366,067
Edges: 33,490
Global components: 332,607


In [111]:
# ============================================================
# GRAPH CONSISTENCY CHECKS
# ============================================================

assert (
    G_2m.number_of_nodes()
    ==
    clusters_2m[
        "local_cluster_id"
    ].nunique()
)

assert (
    G_2m.number_of_edges()
    ==
    len(edges_2m)
)

assert (
    n_components_2m
    <=
    G_2m.number_of_nodes()
)

print("=" * 70)
print("✅ TWO-MONTH GLOBAL GRAPH VALIDATION PASSED")
print("=" * 70)


✅ TWO-MONTH GLOBAL GRAPH VALIDATION PASSED


In [112]:
# ============================================================
# FULL 2021 PRODUCTION TEST
# ============================================================

year_2021_dir = (
    Path("fire_event_production")
    / "2021_full_year"
)

if year_2021_dir.exists():
    shutil.rmtree(year_2021_dir)

year_2021_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(year_2021_dir)

Output directory:
fire_event_production/2021_full_year


In [114]:
# ============================================================
# RUN FULL 2021
# ============================================================

clusters_2021_files, edges_2021_files = (
    process_temporal_range_disk(
        df,
        pd.Timestamp("2021-01-01").date(),
        pd.Timestamp("2021-12-31").date(),
        year_2021_dir,
        verbose=True
    )
)

print("\n" + "=" * 70)
print("FULL 2021 DISK-BACKED ENGINE COMPLETE")
print("=" * 70)

print(
    f"Cluster files: {len(clusters_2021_files)}"
)

print(
    f"Edge files: {len(edges_2021_files)}"
)


DISK-BACKED TEMPORAL ENGINE
Date range: 2021-01-01 → 2021-12-31
Observed dates: 365

[1/365] 2021-01-01
2021-01-01: 2,321 detections, 4,369 spatial pairs
2021-01-01: 3,607 valid pairs
2021-01-01: 1,143 local clusters
2021-01-01: 2,321 detections, 1,143 local clusters

[2/365] 2021-01-02
2021-01-02: 2,035 detections, 2,795 spatial pairs
2021-01-02: 2,350 valid pairs
2021-01-02: 1,152 local clusters
2021-01-02: 2,035 detections, 1,152 local clusters
Comparing previous day 2021-01-01 → 2021-01-02
2021-01-01: 1,191 boundary detections
2021-01-02: 1,324 boundary detections
Cross-day cluster edges: 34
Saved edge file: edges_2021-01-01_2021-01-02.parquet

[3/365] 2021-01-03
2021-01-03: 1,973 detections, 2,845 spatial pairs
2021-01-03: 2,402 valid pairs
2021-01-03: 1,065 local clusters
2021-01-03: 1,973 detections, 1,065 local clusters
Comparing previous day 2021-01-02 → 2021-01-03
2021-01-02: 711 boundary detections
2021-01-03: 1,259 boundary detections
Cross-day cluster edges: 25
Saved edge 

In [115]:
# ============================================================
# 2021 OUTPUT INVENTORY
# ============================================================

print("=" * 70)
print("2021 OUTPUT INVENTORY")
print("=" * 70)

print(
    f"Cluster files: {len(clusters_2021_files)}"
)

print(
    f"Edge files: {len(edges_2021_files)}"
)

print(
    f"Output directory: {year_2021_dir}"
)

print("\nFirst cluster files:")
for f in clusters_2021_files[:5]:
    print(Path(f).name)

print("\nLast cluster files:")
for f in clusters_2021_files[-5:]:
    print(Path(f).name)

print("\nFirst edge files:")
for f in edges_2021_files[:5]:
    print(Path(f).name)

print("\nLast edge files:")
for f in edges_2021_files[-5:]:
    print(Path(f).name)


2021 OUTPUT INVENTORY
Cluster files: 365
Edge files: 341
Output directory: fire_event_production/2021_full_year

First cluster files:
clusters_2021-01-01.parquet
clusters_2021-01-02.parquet
clusters_2021-01-03.parquet
clusters_2021-01-04.parquet
clusters_2021-01-05.parquet

Last cluster files:
clusters_2021-12-27.parquet
clusters_2021-12-28.parquet
clusters_2021-12-29.parquet
clusters_2021-12-30.parquet
clusters_2021-12-31.parquet

First edge files:
edges_2021-01-01_2021-01-02.parquet
edges_2021-01-02_2021-01-03.parquet
edges_2021-01-03_2021-01-04.parquet
edges_2021-01-04_2021-01-05.parquet
edges_2021-01-05_2021-01-06.parquet

Last edge files:
edges_2021-12-26_2021-12-27.parquet
edges_2021-12-27_2021-12-28.parquet
edges_2021-12-28_2021-12-29.parquet
edges_2021-12-29_2021-12-30.parquet
edges_2021-12-30_2021-12-31.parquet


In [116]:
# ============================================================
# PER-FILE CLUSTER VALIDATION
# ============================================================

cluster_file_stats = []

for f in sorted(clusters_2021_files):

    df_tmp = pd.read_parquet(
        f,
        columns=["local_cluster_id"]
    )

    cluster_file_stats.append({
        "file": Path(f).name,
        "n_clusters": len(df_tmp),
        "n_unique": df_tmp["local_cluster_id"].nunique(),
        "n_null": df_tmp["local_cluster_id"].isna().sum()
    })

    assert (
        df_tmp["local_cluster_id"]
        .duplicated()
        .sum()
        == 0
    )

    assert (
        df_tmp["local_cluster_id"]
        .isna()
        .sum()
        == 0
    )

print("=" * 70)
print("2021 CLUSTER FILE VALIDATION")
print("=" * 70)

print(
    f"Files checked: "
    f"{len(cluster_file_stats)}"
)

print(
    f"Total local clusters: "
    f"{sum(x['n_clusters'] for x in cluster_file_stats):,}"
)

print("\n✅ All cluster files passed")


2021 CLUSTER FILE VALIDATION
Files checked: 365
Total local clusters: 700,146

✅ All cluster files passed


In [117]:
# ============================================================
# PER-FILE EDGE VALIDATION
# ============================================================

edge_file_stats = []

for f in sorted(edges_2021_files):

    df_tmp = pd.read_parquet(
        f,
        columns=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )

    edge_file_stats.append({
        "file": Path(f).name,
        "n_edges": len(df_tmp),
        "n_duplicates": df_tmp.duplicated().sum()
    })

    assert (
        df_tmp.duplicated().sum()
        == 0
    )

    assert (
        df_tmp["local_cluster_id_a"]
        .isna()
        .sum()
        == 0
    )

    assert (
        df_tmp["local_cluster_id_b"]
        .isna()
        .sum()
        == 0
    )

print("=" * 70)
print("2021 EDGE FILE VALIDATION")
print("=" * 70)

print(
    f"Files checked: "
    f"{len(edge_file_stats)}"
)

print(
    f"Total cross-day edges: "
    f"{sum(x['n_edges'] for x in edge_file_stats):,}"
)

print("\n✅ All edge files passed")


2021 EDGE FILE VALIDATION
Files checked: 341
Total cross-day edges: 40,916

✅ All edge files passed


In [118]:
# ============================================================
# 2021 GLOBAL CONNECTED COMPONENT PREPARATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("2021 GLOBAL COMPONENT PREPARATION")
print("=" * 70)

# ------------------------------------------------------------
# Load only local cluster IDs from all cluster files
# ------------------------------------------------------------

cluster_id_parts = []

for f in sorted(clusters_2021_files):

    tmp = pd.read_parquet(
        f,
        columns=["local_cluster_id"]
    )

    cluster_id_parts.append(
        tmp["local_cluster_id"]
    )

all_local_cluster_ids = pd.concat(
    cluster_id_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Validate uniqueness
# ------------------------------------------------------------

n_nodes = len(all_local_cluster_ids)
n_unique_nodes = (
    all_local_cluster_ids
    .nunique()
)

print(
    f"Local cluster rows: {n_nodes:,}"
)

print(
    f"Unique local cluster IDs: "
    f"{n_unique_nodes:,}"
)

assert n_nodes == n_unique_nodes

print("\n✅ All local cluster IDs are globally unique")


2021 GLOBAL COMPONENT PREPARATION
Local cluster rows: 700,146
Unique local cluster IDs: 700,146

✅ All local cluster IDs are globally unique


In [119]:
# ============================================================
# STRING ID → INTEGER NODE ID
# ============================================================

node_id = pd.Series(
    np.arange(
        len(all_local_cluster_ids),
        dtype=np.int32
    ),
    index=all_local_cluster_ids.values
)

print("=" * 70)
print("NODE INDEX CREATED")
print("=" * 70)

print(
    f"Number of nodes: "
    f"{len(node_id):,}"
)

print(
    f"Integer index dtype: "
    f"{node_id.dtype}"
)


NODE INDEX CREATED
Number of nodes: 700,146
Integer index dtype: int32


In [120]:
# ============================================================
# UNION-FIND INITIALIZATION
# ============================================================

parent = np.arange(
    n_nodes,
    dtype=np.int32
)

size = np.ones(
    n_nodes,
    dtype=np.int32
)

print("=" * 70)
print("UNION-FIND INITIALIZED")
print("=" * 70)

print(
    f"Nodes: {n_nodes:,}"
)

print(
    f"Parent memory: "
    f"{parent.nbytes / 1024**2:.2f} MB"
)

print(
    f"Size memory: "
    f"{size.nbytes / 1024**2:.2f} MB"
)


UNION-FIND INITIALIZED
Nodes: 700,146
Parent memory: 2.67 MB
Size memory: 2.67 MB


In [121]:
# ============================================================
# UNION-FIND FUNCTIONS
# ============================================================

def find_root(parent, x):
    """
    Find root with path compression.
    """
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]

    return x


def union_nodes(parent, size, a, b):
    """
    Union two nodes using union by size.
    """
    root_a = find_root(parent, a)
    root_b = find_root(parent, b)

    if root_a == root_b:
        return

    # Attach smaller tree to larger tree
    if size[root_a] < size[root_b]:
        root_a, root_b = root_b, root_a

    parent[root_b] = root_a
    size[root_a] += size[root_b]


In [122]:
# ============================================================
# PROCESS ALL 2021 CROSS-DAY EDGES
# ============================================================

print("=" * 70)
print("BUILDING 2021 GLOBAL CONNECTED COMPONENTS")
print("=" * 70)

total_edges_processed = 0

for counter, f in enumerate(
    sorted(edges_2021_files),
    start=1
):

    edges = pd.read_parquet(
        f,
        columns=[
            "local_cluster_id_a",
            "local_cluster_id_b"
        ]
    )

    for row in edges.itertuples(index=False):

        a = node_id[row.local_cluster_id_a]
        b = node_id[row.local_cluster_id_b]

        union_nodes(
            parent,
            size,
            int(a),
            int(b)
        )

    total_edges_processed += len(edges)

    if (
        counter % 25 == 0
        or
        counter == len(edges_2021_files)
    ):
        print(
            f"[{counter}/{len(edges_2021_files)}] "
            f"Edges processed: "
            f"{total_edges_processed:,}"
        )

    del edges

print("\n" + "=" * 70)
print("EDGE PROCESSING COMPLETE")
print("=" * 70)

print(
    f"Total edges processed: "
    f"{total_edges_processed:,}"
)

assert (
    total_edges_processed
    == 40_916
)

print("\n✅ All 2021 cross-day edges processed")


BUILDING 2021 GLOBAL CONNECTED COMPONENTS
[25/341] Edges processed: 801
[50/341] Edges processed: 2,211
[75/341] Edges processed: 14,666
[100/341] Edges processed: 33,655
[125/341] Edges processed: 37,584
[150/341] Edges processed: 38,260
[175/341] Edges processed: 38,523
[200/341] Edges processed: 38,640
[225/341] Edges processed: 38,770
[250/341] Edges processed: 38,905
[275/341] Edges processed: 39,352
[300/341] Edges processed: 39,914
[325/341] Edges processed: 40,501
[341/341] Edges processed: 40,916

EDGE PROCESSING COMPLETE
Total edges processed: 40,916

✅ All 2021 cross-day edges processed


In [123]:
# ============================================================
# FINALIZE ROOTS
# ============================================================

roots = np.empty(
    n_nodes,
    dtype=np.int32
)

for i in range(n_nodes):
    roots[i] = find_root(
        parent,
        i
    )

print("=" * 70)
print("GLOBAL ROOTS CREATED")
print("=" * 70)

print(
    f"Nodes: {len(roots):,}"
)

print(
    f"Unique roots: "
    f"{np.unique(roots).size:,}"
)


GLOBAL ROOTS CREATED
Nodes: 700,146
Unique roots: 659,280


In [126]:
# ============================================================
# CREATE 2021 GLOBAL EVENT IDS
# ============================================================

unique_roots, global_codes = np.unique(
    roots,
    return_inverse=True
)

global_event_ids = (
    global_codes.astype(np.int32)
    + 1
)

print("=" * 70)
print("GLOBAL EVENT IDS CREATED")
print("=" * 70)

print(
    f"Unique roots: {len(unique_roots):,}"
)

print(
    f"Global fire events: "
    f"{len(np.unique(global_event_ids)):,}"
)

print(
    f"Event ID range: "
    f"{global_event_ids.min():,} → "
    f"{global_event_ids.max():,}"
)


GLOBAL EVENT IDS CREATED
Unique roots: 659,280
Global fire events: 659,280
Event ID range: 1 → 659,280


In [127]:
# ============================================================
# LOCAL CLUSTER → GLOBAL FIRE EVENT
# ============================================================

cluster_to_event_2021 = pd.DataFrame({
    "local_cluster_id":
        all_local_cluster_ids.values,

    "global_event_id":
        global_event_ids
})

print("=" * 70)
print("CLUSTER → EVENT MAPPING")
print("=" * 70)

print(
    cluster_to_event_2021.head()
)

print(
    f"\nRows: "
    f"{len(cluster_to_event_2021):,}"
)

print(
    f"Unique local clusters: "
    f"{cluster_to_event_2021['local_cluster_id'].nunique():,}"
)

print(
    f"Unique global events: "
    f"{cluster_to_event_2021['global_event_id'].nunique():,}"
)


CLUSTER → EVENT MAPPING
  local_cluster_id  global_event_id
0       20210101_0                1
1       20210101_1                2
2       20210101_2                3
3       20210101_3                4
4       20210101_4                5

Rows: 700,146
Unique local clusters: 700,146
Unique global events: 659,280


In [128]:
# ============================================================
# GLOBAL COMPONENT VALIDATION
# ============================================================

assert (
    len(cluster_to_event_2021)
    == 700_146
)

assert (
    cluster_to_event_2021[
        "local_cluster_id"
    ].nunique()
    == 700_146
)

assert (
    cluster_to_event_2021[
        "global_event_id"
    ].nunique()
    == len(unique_roots)
)

assert (
    cluster_to_event_2021[
        "global_event_id"
    ].isna()
    .sum()
    == 0
)

print("=" * 70)
print("✅ 2021 GLOBAL COMPONENT VALIDATION PASSED")
print("=" * 70)

print(
    f"Local clusters: "
    f"{700_146:,}"
)

print(
    f"Cross-day edges: "
    f"{40_916:,}"
)

print(
    f"Global fire events: "
    f"{len(unique_roots):,}"
)


✅ 2021 GLOBAL COMPONENT VALIDATION PASSED
Local clusters: 700,146
Cross-day edges: 40,916
Global fire events: 659,280


In [129]:
# ============================================================
# LOCAL CLUSTER → GLOBAL EVENT MAPPING
# ============================================================

cluster_to_event_2021 = pd.DataFrame({
    "local_cluster_id":
        all_local_cluster_ids.values,

    "global_event_id":
        global_event_ids
})

print("=" * 70)
print("CLUSTER → GLOBAL EVENT MAPPING")
print("=" * 70)

print(
    cluster_to_event_2021.head(10)
)

print(
    f"\nRows: "
    f"{len(cluster_to_event_2021):,}"
)

print(
    f"Unique local clusters: "
    f"{cluster_to_event_2021['local_cluster_id'].nunique():,}"
)

print(
    f"Unique global events: "
    f"{cluster_to_event_2021['global_event_id'].nunique():,}"
)


CLUSTER → GLOBAL EVENT MAPPING
  local_cluster_id  global_event_id
0       20210101_0                1
1       20210101_1                2
2       20210101_2                3
3       20210101_3                4
4       20210101_4                5
5       20210101_5                6
6       20210101_6                7
7       20210101_7                8
8       20210101_8                9
9       20210101_9               10

Rows: 700,146
Unique local clusters: 700,146
Unique global events: 659,280


In [130]:
# ============================================================
# SAVE 2021 GLOBAL EVENT MAPPING
# ============================================================

mapping_path = (
    year_2021_dir
    / "global_cluster_to_event_2021.parquet"
)

cluster_to_event_2021.to_parquet(
    mapping_path,
    index=False
)

print("=" * 70)
print("GLOBAL EVENT MAPPING SAVED")
print("=" * 70)

print(
    f"Path: {mapping_path}"
)

print(
    f"Exists: {mapping_path.exists()}"
)


GLOBAL EVENT MAPPING SAVED
Path: fire_event_production/2021_full_year/global_cluster_to_event_2021.parquet
Exists: True


In [131]:
# ============================================================
# 2021 GLOBAL EVENT STRUCTURE
# ============================================================

clusters_per_event = (
    cluster_to_event_2021
    .groupby("global_event_id")
    .size()
)

print("=" * 70)
print("2021 GLOBAL EVENT STRUCTURE")
print("=" * 70)

print(
    f"Global events: "
    f"{len(clusters_per_event):,}"
)

print(
    f"Events containing >1 local cluster: "
    f"{(clusters_per_event > 1).sum():,}"
)

print(
    f"Maximum local clusters in one event: "
    f"{clusters_per_event.max():,}"
)

print("\nDistribution:")
print(
    clusters_per_event.describe()
)


2021 GLOBAL EVENT STRUCTURE
Global events: 659,280
Events containing >1 local cluster: 28,706
Maximum local clusters in one event: 49

Distribution:
count    659280.000000
mean          1.061986
std           0.404210
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          49.000000
dtype: float64


In [132]:
merged_clusters = (
    clusters_per_event[
        clusters_per_event > 1
    ]
    .sum()
)

print(
    f"\nLocal clusters participating in "
    f"multi-cluster events: "
    f"{merged_clusters:,}"
)

print(
    f"Local clusters remaining as "
    f"single-cluster events: "
    f"{(clusters_per_event == 1).sum():,}"
)



Local clusters participating in multi-cluster events: 69,572
Local clusters remaining as single-cluster events: 630,574


In [133]:
YEAR_2021_DIR = Path(
    "/content/fire_event_production/2021_full_year"
)

cluster_2021_files = sorted(
    YEAR_2021_DIR.glob("clusters_*.parquet")
)

edge_2021_files = sorted(
    YEAR_2021_DIR.glob("edges_*.parquet")
)

print("2021 cluster files:", len(cluster_2021_files))
print("2021 edge files:", len(edge_2021_files))


2021 cluster files: 0
2021 edge files: 0


In [134]:
from pathlib import Path

YEAR_DIR = Path("fire_event_production") / "2021_full_year"

cluster_2021_files = sorted(
    YEAR_DIR.glob("clusters_*.parquet")
)

edge_2021_files = sorted(
    YEAR_DIR.glob("edges_*.parquet")
)

print("=" * 70)
print("2021 FILE INVENTORY")
print("=" * 70)

print("2021 cluster files:", len(cluster_2021_files))
print("2021 edge files:", len(edge_2021_files))

print("\nFirst cluster:")
print(cluster_2021_files[0] if cluster_2021_files else "NONE")

print("\nLast cluster:")
print(cluster_2021_files[-1] if cluster_2021_files else "NONE")

print("\nFirst edge:")
print(edge_2021_files[0] if edge_2021_files else "NONE")

print("\nLast edge:")
print(edge_2021_files[-1] if edge_2021_files else "NONE")


2021 FILE INVENTORY
2021 cluster files: 0
2021 edge files: 0

First cluster:
NONE

Last cluster:
NONE

First edge:
NONE

Last edge:
NONE


In [2]:
import os
print(os.listdir("/content/"))

['default_configs.db', 'active_config', 'gce', '.last_opt_in_prompt.yaml', 'configurations', '.last_update_check.json', 'logs', 'config_sentinel', 'hidden_gcloud_config_universe_descriptor_data_cache_configs.db', '.last_survey_prompt.yaml']
